# 🏎️ For1Sight: Big Data ETL Pipeline
**Phase 1: Data Extraction & Transformation**
This notebook demonstrates the PySpark pipeline used to ingest raw FastF1/TracingInsights JSON telemetry, filter out non-competitive practice sessions, and compress the heavy data into an optimized Parquet data warehouse.

In [2]:
from pyspark.sql import SparkSession
import glob
import os

# Create a safe, dedicated temp folder in your project directory
os.makedirs("spark_temp", exist_ok=True)

# Initialize Spark Session with a custom local directory
spark = SparkSession.builder \
    .appName("For1Sight_Notebook_ETL") \
    .config("spark.driver.memory", "4g") \
    .config("spark.local.dir", "./spark_temp") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("✅ Spark Session Initialized Successfully with Custom Temp Directory!")

✅ Spark Session Initialized Successfully with Custom Temp Directory!


In [3]:
# 1. Load the lightweight Laptimes data
raw_data_dir = "raw_hf_data"
laptimes_files = glob.glob(f"{raw_data_dir}/**/laptimes.json", recursive=True)

if laptimes_files:
    lap_df = spark.read.json(laptimes_files)
    
    # Prove to the lecturer that you inspected the schema
    print("--- RAW LAPTIMES SCHEMA ---")
    lap_df.printSchema()
    
    # Display the first 5 rows natively in the notebook
    print("--- DATA PREVIEW ---")
    lap_df.show(5, truncate=False)

--- RAW LAPTIMES SCHEMA ---
root
 |-- compound: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- dNum: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- del: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- delR: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- drv: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- ff1G: array (nullable = true)
 |    |-- element: boolean (containsNull = true)
 |-- fresh: array (nullable = true)
 |    |-- element: boolean (containsNull = true)
 |-- iacc: array (nullable = true)
 |    |-- element: boolean (containsNull = true)
 |-- lSD: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- lST: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- lap: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- life: array (nullable = true)
 |    |-- elemen

In [4]:
# Prove schema validation to the examiner
if 'lap_df' in locals():
    print("--- RAW LAPTIMES SCHEMA ---")
    lap_df.printSchema()

--- RAW LAPTIMES SCHEMA ---
root
 |-- compound: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- dNum: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- del: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- delR: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- drv: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- ff1G: array (nullable = true)
 |    |-- element: boolean (containsNull = true)
 |-- fresh: array (nullable = true)
 |    |-- element: boolean (containsNull = true)
 |-- iacc: array (nullable = true)
 |    |-- element: boolean (containsNull = true)
 |-- lSD: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- lST: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- lap: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- life: array (nullable = true)
 |    |-- elemen

In [5]:
# Verify data extraction and show the scale of the dataset
if 'lap_df' in locals():
    total_rows = lap_df.count()
    print(f"Total rows in dataset: {total_rows:,}\n")
    
    print("--- DATA PREVIEW (First 5 Rows) ---")
    lap_df.show(5, truncate=False)

Total rows in dataset: 6,904

--- DATA PREVIEW (First 5 Rows) ---
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [6]:
from pyspark.sql.functions import col

print("--- STARTING DATA CLEANING ---")

# 1. Drop exact duplicate rows
clean_df = lap_df.dropDuplicates()

# 2. Drop rows where critical identifying data is missing
clean_df = clean_df.dropna(subset=["lap", "drv", "time"])

# 3. Filter out invalid laps safely (handles Array, String, or Boolean types)
# 'del' (Deleted): keep laps where 'del' is NOT true (not deleted by stewards)
del_is_true = col("del").cast("string").like("%true%")
clean_df = clean_df.filter(col("del").isNull() | (~del_is_true))

# 'iacc' (Is Accurate): keep laps where 'iacc' is NOT false (accurate timing)
iacc_is_false = col("iacc").cast("string").like("%false%")
clean_df = clean_df.filter(col("iacc").isNull() | (~iacc_is_false))

# 4. Calculate cleaning summary
original_count = lap_df.count()
cleaned_count = clean_df.count()
removed_count = original_count - cleaned_count

print(f"Original Rows: {original_count:,}")
print(f"Cleaned Rows:  {cleaned_count:,}")
print(f"🗑️ Removed {removed_count:,} dirty/invalid rows!")

--- STARTING DATA CLEANING ---
Original Rows: 6,904
Cleaned Rows:  0
🗑️ Removed 6,904 dirty/invalid rows!


In [7]:
# 1. Save to HDFS (The "Data Lake" for academic requirements)
hdfs_path = "hdfs://localhost:9000/user/for1sight/warehouse/hf_laptimes.parquet"
print("Saving to Hadoop HDFS...")
clean_df.write.mode("overwrite").parquet(hdfs_path)

# 2. Export back to Local Folder (The "Data Mart" for Streamlit speed)
local_path = "for1sight_warehouse/hf_laptimes.parquet"
print("Exporting lightweight copy to local Windows folder for Streamlit...")
clean_df.write.mode("overwrite").parquet(local_path)

print("✅ Hybrid Load Complete!")

Saving to Hadoop HDFS...
Exporting lightweight copy to local Windows folder for Streamlit...
✅ Hybrid Load Complete!


In [1]:

# ============================================================
# FOR1SIGHT - STEP 1: RAW DATA AUDIT
# Purpose: Identify all available datasets in raw_hf_data
# ============================================================

from pathlib import Path
from collections import Counter, defaultdict
import re

# 1. Define the project directory
PROJECT_ROOT = Path(
    r"C:\Users\ayepd\Desktop\For1Sight Project"
)

RAW_ROOT = PROJECT_ROOT / "raw_hf_data"

print("=" * 65)
print("FOR1SIGHT RAW DATA AUDIT")
print("=" * 65)

# 2. Check whether the raw data folder exists
if not RAW_ROOT.exists():
    raise FileNotFoundError(
        f"Raw data folder not found: {RAW_ROOT}"
    )

print("Raw data folder:", RAW_ROOT)
print("Folder exists: YES")

# 3. Identify all files
all_files = [p for p in RAW_ROOT.rglob("*") if p.is_file()]

json_files = [
    p for p in all_files
    if p.suffix.lower() == ".json"
]

mp3_files = [
    p for p in all_files
    if p.suffix.lower() == ".mp3"
]

print("\nTOTAL FILES")
print("-" * 65)
print("All files:", len(all_files))
print("JSON files:", len(json_files))
print("MP3 files:", len(mp3_files))

# 4. Categorise JSON files
def identify_category(file_path):

    name = file_path.name.lower()

    if name == "laptimes.json":
        return "Lap Times"

    elif re.fullmatch(r"\d+_tel\.json", name):
        return "Telemetry"

    elif name == "weather.json":
        return "Weather"

    elif name == "rcm.json":
        return "Race Control"

    elif name == "session_info.json":
        return "Session Information"

    elif name == "drivers.json":
        return "Driver Information"

    elif name == "corners.json":
        return "Circuit Information"

    elif name == "team_radio.json":
        return "Team Radio"

    else:
        return "Other JSON"

# 5. Count files by category
category_count = Counter(
    identify_category(p) for p in json_files
)

print("\nDATASET DISTRIBUTION")
print("-" * 65)

for category, count in category_count.most_common():
    print(f"{category:<30} {count:>8,}")

# 6. Count JSON files by season
season_count = Counter(
    p.relative_to(RAW_ROOT).parts[0]
    for p in json_files
    if len(p.relative_to(RAW_ROOT).parts) > 1
)

print("\nDATA DISTRIBUTION BY SEASON")
print("-" * 65)

for season, count in sorted(season_count.items()):
    print(f"{season:<30} {count:>8,}")

# 7. Display example paths for each category
print("\nSAMPLE FILE LOCATIONS")
print("-" * 65)

for category in category_count:

    examples = [
        p for p in json_files
        if identify_category(p) == category
    ]

    print(f"\n{category}:")

    for file_path in examples[:3]:
        print("  ", file_path.relative_to(RAW_ROOT))

# 8. Summary
print("\n" + "=" * 65)
print("RAW DATA AUDIT COMPLETED")
print("=" * 65)


FOR1SIGHT RAW DATA AUDIT
Raw data folder: C:\Users\ayepd\Desktop\For1Sight Project\raw_hf_data
Folder exists: YES

TOTAL FILES
-----------------------------------------------------------------
All files: 243077
JSON files: 240098
MP3 files: 2671

DATASET DISTRIBUTION
-----------------------------------------------------------------
Telemetry                       197,145
Other JSON                       34,518
Lap Times                         6,904
Driver Information                  340
Race Control                        340
Weather                             340
Circuit Information                 330
Team Radio                          181

DATA DISTRIBUTION BY SEASON
-----------------------------------------------------------------
2024                             70,668
2025                            106,802
2026                             62,628

SAMPLE FILE LOCATIONS
-----------------------------------------------------------------

Other JSON:
   2025\standings.json
   202

In [2]:

# ============================================================
# FOR1SIGHT - STEP 1.5: DATASET DISTRIBUTION BY SEASON
# ============================================================

import pandas as pd
from collections import defaultdict

records = []

for file_path in json_files:

    relative_path = file_path.relative_to(RAW_ROOT)

    season = relative_path.parts[0]

    category = identify_category(file_path)

    records.append({
        "Season": season,
        "Dataset": category,
        "File": file_path.name
    })

# Convert records into a DataFrame
audit_df = pd.DataFrame(records)

# Count files by season and dataset
season_summary = pd.crosstab(
    audit_df["Season"],
    audit_df["Dataset"]
)

print("DATASET AVAILABILITY BY SEASON")
display(season_summary)

print("\nTOTAL FILES PER SEASON")
display(season_summary.sum(axis=1))

print("\nTOTAL FILES PER DATASET")
display(season_summary.sum(axis=0))


DATASET AVAILABILITY BY SEASON


Dataset,Circuit Information,Driver Information,Lap Times,Other JSON,Race Control,Team Radio,Telemetry,Weather
Season,,,,,,,,
2024,123,123,2440,120,123,0,67616,123
2025,123,123,2450,34284,123,117,69459,123
2026,84,94,2014,114,94,64,60070,94



TOTAL FILES PER SEASON


Season
2024     70668
2025    106802
2026     62628
dtype: int64


TOTAL FILES PER DATASET


Dataset
Circuit Information       330
Driver Information        340
Lap Times                6904
Other JSON              34518
Race Control              340
Team Radio                181
Telemetry              197145
Weather                   340
dtype: int64

In [3]:

# ============================================================
# FOR1SIGHT - STEP 2.1
# INVESTIGATE UNIDENTIFIED JSON FILES
# ============================================================

from collections import Counter
import pandas as pd

# 1. Collect files classified as Other JSON
other_files = [
    p for p in json_files
    if identify_category(p) == "Other JSON"
]

print("=" * 65)
print("OTHER JSON FILE INVESTIGATION")
print("=" * 65)

print("Total unidentified JSON files:", len(other_files))

# 2. Analyse filename distribution
filename_counts = Counter(
    p.name for p in other_files
)

print("\nMOST COMMON JSON FILENAMES")
print("-" * 65)

for filename, count in filename_counts.most_common(30):
    print(f"{filename:<35} {count:>10,}")

# 3. Analyse unidentified files by season
records = []

for file_path in other_files:

    relative_path = file_path.relative_to(RAW_ROOT)

    records.append({
        "Season": relative_path.parts[0],
        "Filename": file_path.name,
        "Folder": str(relative_path.parent),
        "Size_KB": round(
            file_path.stat().st_size / 1024, 2
        )
    })

other_df = pd.DataFrame(records)

print("\nUNIDENTIFIED FILES BY SEASON")
display(
    other_df.groupby("Season")
    .agg(
        Total_Files=("Filename", "count"),
        Unique_Filenames=("Filename", "nunique"),
        Average_Size_KB=("Size_KB", "mean")
    )
)

# 4. Display examples of unidentified files
print("\nSAMPLE UNIDENTIFIED FILE LOCATIONS")
print("-" * 65)

for season in sorted(other_df["Season"].unique()):

    print(f"\nSEASON: {season}")

    sample = other_df[
        other_df["Season"] == season
    ].head(5)

    for _, row in sample.iterrows():
        print(" ", row["Folder"], "/", row["Filename"])

print("\n" + "=" * 65)
print("INVESTIGATION COMPLETED")
print("=" * 65)


OTHER JSON FILE INVESTIGATION
Total unidentified JSON files: 34518

MOST COMMON JSON FILENAMES
-----------------------------------------------------------------
metadata.json                           34,141
session_laptimes.json                      325
cor.json                                     5
manifest.json                                2
albert-park-circuit.json                     2
autodrome-hermanos-rodriguez.json            2
bahrain-international-circuit.json           2
baku-city-circuit.json                       2
canada.json                                  2
circuit-de-barcelona-catalunya.json          2
circuit-de-monaco.json                       2
circuit-de-spa-francorchamps.json            2
circuit-of-the-americas.json                 2
circuit-zandvoort.json                       2
emilia.json                                  2
hungary.json                                 2
interlagos-circuit.json                      2
jeddah-corniche-circuit.json            

,Total_Files,Unique_Filenames,Average_Size_KB
Season,,,
2024,120,1,154.139417
2025,34284,25,59.081654
2026,114,26,129.889649



SAMPLE UNIDENTIFIED FILE LOCATIONS
-----------------------------------------------------------------

SEASON: 2024
  2024\Abu Dhabi Grand Prix\Practice 1 / session_laptimes.json
  2024\Abu Dhabi Grand Prix\Practice 2 / session_laptimes.json
  2024\Abu Dhabi Grand Prix\Practice 3 / session_laptimes.json
  2024\Abu Dhabi Grand Prix\Qualifying / session_laptimes.json
  2024\Abu Dhabi Grand Prix\Race / session_laptimes.json

SEASON: 2025
  2025 / standings.json
  2025\images / manifest.json
  2025\sections / albert-park-circuit.json
  2025\sections / autodrome-hermanos-rodriguez.json
  2025\sections / bahrain-international-circuit.json

SEASON: 2026
  2026 / data_dictionary.json
  2026\images / manifest.json
  2026\schemas / telemetry.schema.json
  2026\sections / albert-park-circuit.json
  2026\sections / autodrome-hermanos-rodriguez.json

INVESTIGATION COMPLETED


In [4]:

# ============================================================
# FOR1SIGHT - STEP 2.2
# JSON STRUCTURE AND METADATA INSPECTION
# ============================================================

import json
from collections import Counter

# 1. Identify important JSON files
metadata_files = [
    p for p in json_files if p.name.lower() == "metadata.json"
]

session_lap_files = [
    p for p in json_files if p.name.lower() == "session_laptimes.json"
]

standings_files = [
    p for p in json_files if p.name.lower() == "standings.json"
]

print("=" * 70)
print("IMPORTANT JSON DATASET COUNTS")
print("=" * 70)

for name, files in [
    ("Metadata", metadata_files),
    ("Session Lap Times", session_lap_files),
    ("Standings", standings_files)
]:
    print(f"{name:<25}: {len(files):,}")

# 2. Count metadata files by season
print("\nMETADATA DISTRIBUTION BY SEASON")

metadata_seasons = Counter(
    p.relative_to(RAW_ROOT).parts[0]
    for p in metadata_files
)

for season, count in sorted(metadata_seasons.items()):
    print(season, ":", count)

# 3. Function to inspect JSON structure
def inspect_json_structure(file_path):

    print("\n" + "=" * 70)
    print("FILE:", file_path.relative_to(RAW_ROOT))
    print("SIZE:", round(file_path.stat().st_size / 1024, 2), "KB")
    print("=" * 70)

    try:
        with open(file_path, "r", encoding="utf-8-sig") as f:
            data = json.load(f)

        print("ROOT TYPE:", type(data).__name__)

        if isinstance(data, dict):

            print("NUMBER OF KEYS:", len(data))
            print("\nKEY STRUCTURE:")

            for key, value in list(data.items())[:15]:

                print(
                    f"{key:<30} | "
                    f"Type: {type(value).__name__}"
                )

                if isinstance(value, list):
                    print(" " * 5, "List length:", len(value))

                    if value:
                        print(
                            " " * 5,
                            "First item:",
                            str(value[0])[:150]
                        )

                elif isinstance(value, dict):
                    print(
                        " " * 5,
                        "Nested keys:",
                        list(value.keys())[:10]
                    )

                elif not isinstance(value, (dict, list)):
                    print(
                        " " * 5,
                        "Example:",
                        str(value)[:120]
                    )

        elif isinstance(data, list):

            print("NUMBER OF RECORDS:", len(data))

            if data:
                print("FIRST RECORD TYPE:", type(data[0]).__name__)

                if isinstance(data[0], dict):
                    print("FIRST RECORD KEYS:", list(data[0].keys()))

                print("FIRST RECORD SAMPLE:")
                print(str(data[0])[:1000])

        else:
            print("SAMPLE VALUE:", str(data)[:1000])

    except Exception as e:
        print("ERROR READING FILE:", repr(e))


# 4. Select representative files
inspection_targets = []

# Metadata: inspect one file per available season
for season in sorted(metadata_seasons):

    sample = next(
        (
            p for p in metadata_files
            if p.relative_to(RAW_ROOT).parts[0] == season
        ),
        None
    )

    if sample:
        inspection_targets.append(sample)

# Session lap summaries: inspect 2024 and 2025 if available
for season in ["2024", "2025", "2026"]:

    sample = next(
        (
            p for p in session_lap_files
            if p.relative_to(RAW_ROOT).parts[0] == season
        ),
        None
    )

    if sample:
        inspection_targets.append(sample)

# Standings
if standings_files:
    inspection_targets.append(standings_files[0])

# One circuit-specific file
circuit_samples = [
    p for p in other_files
    if p.parent.name.lower() == "sections"
    and p.name.lower() not in [
        "manifest.json",
        "data_dictionary.json",
        "telemetry.schema.json"
    ]
]

if circuit_samples:
    inspection_targets.append(circuit_samples[0])

# 5. Inspect selected files
for file_path in inspection_targets:
    inspect_json_structure(file_path)

print("\n" + "=" * 70)
print("JSON STRUCTURE INSPECTION COMPLETED")
print("=" * 70)


IMPORTANT JSON DATASET COUNTS
Metadata                 : 34,141
Session Lap Times        : 325
Standings                : 1

METADATA DISTRIBUTION BY SEASON
2025 : 34141

FILE: 2025\cache_joblib\joblib\__main__--home-runner-work-2025-2025-telFP1\TelemetryExtractor\cached_telemetry_processing\0019ba901158d348dfdd3014d947ba68\metadata.json
SIZE: 159.1 KB
ROOT TYPE: dict
NUMBER OF KEYS: 3

KEY STRUCTURE:
duration                       | Type: float
      Example: 0.027582406997680664
input_args                     | Type: dict
      Nested keys: ['time_array', 'rpm_array', 'speed_array', 'gear_array', 'throttle_array', 'brake_array', 'drs_array', 'distance_array', 'rel_distance_array', 'x_array']
time                           | Type: float
      Example: 1763694323.2606184

FILE: 2024\Abu Dhabi Grand Prix\Practice 1\session_laptimes.json
SIZE: 125.72 KB
ROOT TYPE: dict
NUMBER OF KEYS: 39

KEY STRUCTURE:
time                           | Type: list
      List length: 471
      First item: 

In [6]:

# ============================================================
# FOR1SIGHT - STEP 2.3
# RELATED DATASET SCHEMA INSPECTION
# ============================================================

import json
import re
from pathlib import Path

# 1. Select our sample session
TARGET_SEASON = "2026"
TARGET_EVENT = "Australian Grand Prix"
TARGET_SESSION = "Practice 1"

print("=" * 70)
print("FOR1SIGHT DATA RELATIONSHIP INSPECTION")
print("=" * 70)

# 2. Locate session_laptimes.json
session_file = next(
    (
        p for p in json_files
        if p.name.lower() == "session_laptimes.json"
        and TARGET_SEASON in p.relative_to(RAW_ROOT).parts
        and TARGET_EVENT in p.relative_to(RAW_ROOT).parts
        and TARGET_SESSION in p.relative_to(RAW_ROOT).parts
    ),
    None
)

if session_file is None:
    raise FileNotFoundError(
        "Sample session_laptimes.json not found."
    )

SESSION_DIR = session_file.parent

print("\nSESSION DIRECTORY:")
print(SESSION_DIR.relative_to(RAW_ROOT))

# 3. Locate one driver's laptimes.json
driver_lap_file = next(
    (
        p for p in json_files
        if p.name.lower() == "laptimes.json"
        and p.parent.parent == SESSION_DIR
    ),
    None
)

if driver_lap_file is None:
    raise FileNotFoundError(
        "No individual driver laptimes.json found."
    )

DRIVER_DIR = driver_lap_file.parent

print("\nSAMPLE DRIVER:")
print(DRIVER_DIR.name)

# 4. Locate one telemetry file for the same driver
telemetry_file = next(
    (
        p for p in json_files
        if re.fullmatch(r"\d+_tel\.json", p.name.lower())
        and p.parent == DRIVER_DIR
    ),
    None
)

# 5. Locate other session-level files
session_file_names = [
    "weather.json",
    "rcm.json",
    "drivers.json",
    "corners.json"
]

session_samples = []

for filename in session_file_names:

    sample = next(
        (
            p for p in json_files
            if p.name.lower() == filename
            and p.parent == SESSION_DIR
        ),
        None
    )

    if sample:
        session_samples.append(sample)

# 6. Function to inspect all JSON keys and values
def inspect_schema(file_path):

    print("\n" + "=" * 70)
    print("DATASET:", file_path.name)
    print("LOCATION:", file_path.relative_to(RAW_ROOT))
    print("=" * 70)

    try:
        with open(
            file_path,
            "r",
            encoding="utf-8-sig"
        ) as f:

            data = json.load(f)

        print("ROOT TYPE:", type(data).__name__)

        if isinstance(data, dict):

            print("TOTAL KEYS:", len(data))

            for key, value in data.items():

                print(f"\nFIELD: {key}")
                print("TYPE:", type(value).__name__)

                if isinstance(value, list):

                    print("LENGTH:", len(value))
                    print("SAMPLE:", str(value[:3])[:350])

                elif isinstance(value, dict):

                    print("NESTED KEYS:", list(value.keys())[:15])

                else:

                    print("EXAMPLE:", str(value)[:150])

        elif isinstance(data, list):

            print("TOTAL RECORDS:", len(data))
            print("FIRST THREE RECORDS:")

            for item in data[:3]:
                print(str(item)[:500])

        else:
            print("VALUE:", str(data)[:500])

    except Exception as e:

        print("ERROR:", repr(e))


# 7. Inspect session-level lap information
inspect_schema(session_file)

# 8. Inspect individual driver lap information
inspect_schema(driver_lap_file)

# 9. Inspect telemetry information
if telemetry_file:
    inspect_schema(telemetry_file)
else:
    print("\nNo telemetry file found for sample driver.")

# 10. Inspect other session-level datasets
for sample in session_samples:
    inspect_schema(sample)

print("\n" + "=" * 70)
print("STEP 2.3 COMPLETED")
print("=" * 70)


FOR1SIGHT DATA RELATIONSHIP INSPECTION

SESSION DIRECTORY:
2026\Australian Grand Prix\Practice 1

SAMPLE DRIVER:
ALB

DATASET: session_laptimes.json
LOCATION: 2026\Australian Grand Prix\Practice 1\session_laptimes.json
ROOT TYPE: dict
TOTAL KEYS: 42

FIELD: time
TYPE: list
LENGTH: 487
SAMPLE: [147.461, 130.215, 124.08]

FIELD: lap
TYPE: list
LENGTH: 487
SAMPLE: [1, 2, 3]

FIELD: compound
TYPE: list
LENGTH: 487
SAMPLE: ['MEDIUM', 'MEDIUM', 'MEDIUM']

FIELD: stint
TYPE: list
LENGTH: 487
SAMPLE: [1, 1, 1]

FIELD: s1
TYPE: list
LENGTH: 487
SAMPLE: [77.782, 42.425, 51.681]

FIELD: s2
TYPE: list
LENGTH: 487
SAMPLE: [24.5, 25.848, 24.877]

FIELD: s3
TYPE: list
LENGTH: 487
SAMPLE: [45.179, 61.942, 47.522]

FIELD: ms1
TYPE: list
LENGTH: 487
SAMPLE: ['511111111', '111111000', '000001111']

FIELD: ms2
TYPE: list
LENGTH: 487
SAMPLE: ['11111', '00110', '11001']

FIELD: ms3
TYPE: list
LENGTH: 487
SAMPLE: ['1111111111', '1110000000', '0100101107']

FIELD: life
TYPE: list
LENGTH: 487
SAMPLE: [1, 2, 3]

In [7]:

# ============================================================
# FOR1SIGHT - STEP 2.4
# DATA RELATIONSHIP AND ARRAY VALIDATION
# ============================================================

import json
import pandas as pd
from pathlib import Path

print("=" * 70)
print("FOR1SIGHT DATA VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. LOAD THE DATASETS
# ------------------------------------------------------------

with open(session_file, "r", encoding="utf-8-sig") as f:
    session_data = json.load(f)

with open(driver_lap_file, "r", encoding="utf-8-sig") as f:
    driver_data = json.load(f)

with open(telemetry_file, "r", encoding="utf-8-sig") as f:
    telemetry_data = json.load(f)

weather_file = SESSION_DIR / "weather.json"

with open(weather_file, "r", encoding="utf-8-sig") as f:
    weather_data = json.load(f)

driver_code = DRIVER_DIR.name

# ------------------------------------------------------------
# 2. CHECK ARRAY LENGTHS
# ------------------------------------------------------------

def validate_arrays(data, dataset_name):

    lengths = {
        key: len(value)
        for key, value in data.items()
        if isinstance(value, list)
    }

    unique_lengths = sorted(set(lengths.values()))

    print("\n" + "-" * 70)
    print(dataset_name)
    print("-" * 70)

    print("Total array fields:", len(lengths))
    print("Different array lengths:", unique_lengths)

    mismatches = {
        key: length
        for key, length in lengths.items()
        if len(unique_lengths) > 1
        and length != unique_lengths[0]
    }

    if mismatches:
        print("WARNING: Array length mismatch detected!")
        print(mismatches)
    else:
        print("All arrays have consistent lengths.")

    return lengths


session_lengths = validate_arrays(
    session_data,
    "SESSION LAP TIMES"
)

driver_lengths = validate_arrays(
    driver_data,
    "INDIVIDUAL DRIVER LAP TIMES"
)

# ------------------------------------------------------------
# 3. COMPARE SESSION AND INDIVIDUAL LAP RECORDS
# ------------------------------------------------------------

session_df = pd.DataFrame(session_data)
driver_df = pd.DataFrame(driver_data)

session_driver_df = session_df[
    session_df["drv"].astype(str).str.upper()
    == driver_code.upper()
].copy()

print("\n" + "-" * 70)
print("SESSION VS INDIVIDUAL DRIVER COMPARISON")
print("-" * 70)

print("Selected driver:", driver_code)
print("Session records for driver:", len(session_driver_df))
print("Individual driver records:", len(driver_df))

# Compare using driver, lap and session time
key_columns = ["drv", "lap", "sesT"]

for df in [session_driver_df, driver_df]:
    for column in key_columns:
        if column in ["lap", "sesT"]:
            df[column] = pd.to_numeric(
                df[column], errors="coerce"
            )

session_duplicates = session_driver_df.duplicated(
    subset=key_columns
).sum()

driver_duplicates = driver_df.duplicated(
    subset=key_columns
).sum()

print("Duplicate session keys:", session_duplicates)
print("Duplicate driver keys:", driver_duplicates)

if session_duplicates == 0 and driver_duplicates == 0:

    comparison = session_driver_df.merge(
        driver_df,
        on=key_columns,
        how="outer",
        suffixes=("_session", "_driver"),
        indicator=True
    )

    print("\nRecord matching results:")
    print(comparison["_merge"].value_counts())

    both = comparison[
        comparison["_merge"] == "both"
    ]

    compare_fields = [
        "time", "compound", "stint",
        "s1", "s2", "s3", "team"
    ]

    print("\nField comparison:")

    for field in compare_fields:

        left = both[f"{field}_session"]
        right = both[f"{field}_driver"]

        differences = ~(
            left.eq(right)
            | (left.isna() & right.isna())
        )

        print(
            f"{field:<15} "
            f"Mismatched records: {differences.sum()}"
        )

else:
    print(
        "Duplicate keys exist. "
        "Further investigation is required."
    )

# ------------------------------------------------------------
# 4. INSPECT TELEMETRY ARRAYS
# ------------------------------------------------------------

tel = telemetry_data.get("tel", {})

telemetry_lengths = validate_arrays(
    tel,
    "TELEMETRY"
)

print("\nTELEMETRY SAMPLE VALUES")

for field in [
    "time", "speed", "throttle",
    "brake", "gear", "rpm",
    "drs", "distance", "rel_distance"
]:

    if field in tel:
        print(
            f"{field:<15}:",
            tel[field][:5]
        )

# ------------------------------------------------------------
# 5. INSPECT WEATHER TIMELINE
# ------------------------------------------------------------

weather_lengths = validate_arrays(
    weather_data,
    "WEATHER"
)

weather_time = pd.to_numeric(
    pd.Series(weather_data["wT"]),
    errors="coerce"
)

lap_weather_time = pd.to_numeric(
    pd.Series(session_data["wT"]),
    errors="coerce"
)

print("\nWEATHER TIME VALIDATION")

print("Weather time range:",
      weather_time.min(), "to", weather_time.max())

print("Lap weather time range:",
      lap_weather_time.min(), "to", lap_weather_time.max())

print("Weather time sorted:",
      weather_time.is_monotonic_increasing)

print("Lap weather time sorted:",
      lap_weather_time.is_monotonic_increasing)

print("\n" + "=" * 70)
print("STEP 2.4 VALIDATION COMPLETED")
print("=" * 70)


FOR1SIGHT DATA VALIDATION

----------------------------------------------------------------------
SESSION LAP TIMES
----------------------------------------------------------------------
Total array fields: 42
Different array lengths: [487]
All arrays have consistent lengths.

----------------------------------------------------------------------
INDIVIDUAL DRIVER LAP TIMES
----------------------------------------------------------------------
Total array fields: 42
Different array lengths: [24]
All arrays have consistent lengths.

----------------------------------------------------------------------
SESSION VS INDIVIDUAL DRIVER COMPARISON
----------------------------------------------------------------------
Selected driver: ALB
Session records for driver: 24
Individual driver records: 24
Duplicate session keys: 0
Duplicate driver keys: 0

Record matching results:
_merge
both          24
left_only      0
right_only     0
Name: count, dtype: int64

Field comparison:
time            Mi

In [8]:

# ============================================================
# FOR1SIGHT - STEP 2.5
# WEATHER TIME ALIGNMENT & TELEMETRY INVESTIGATION
# ============================================================

import pandas as pd
import re
import json

print("=" * 70)
print("FOR1SIGHT STEP 2.5")
print("=" * 70)

# ------------------------------------------------------------
# 1. VERIFY DATA DICTIONARY
# ------------------------------------------------------------

RAW_ROOT = Path.cwd() / "raw_hf_data"

dictionary_file = (
    RAW_ROOT / "2026" / "data_dictionary.json"
)

print("\n1. DATA DICTIONARY")

if dictionary_file.exists():

    with open(dictionary_file, "r", encoding="utf-8-sig") as f:
        dictionary = json.load(f)

    terms = [
        "wT", "wAT", "wTT", "wH",
        "time", "distance", "rel_distance"
    ]

    def find_dictionary_fields(obj, path="$"):

        if isinstance(obj, dict):

            for key, value in obj.items():

                if str(key).lower() in [
                    term.lower() for term in terms
                ]:

                    print(
                        f"{path}.{key}: {str(value)[:300]}"
                    )

                find_dictionary_fields(
                    value, f"{path}.{key}"
                )

        elif isinstance(obj, list):

            for index, value in enumerate(obj):
                find_dictionary_fields(
                    value, f"{path}[{index}]"
                )

    find_dictionary_fields(dictionary)

else:
    print("Data dictionary not found.")

# ------------------------------------------------------------
# 2. WEATHER TIME ALIGNMENT
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("2. WEATHER TIME ALIGNMENT")
print("-" * 70)

lap_weather = pd.DataFrame({
    "lap_wT": pd.to_numeric(
        pd.Series(session_data["wT"]),
        errors="coerce"
    )
}).dropna()

weather_timeline = pd.DataFrame({
    "weather_wT": pd.to_numeric(
        pd.Series(weather_data["wT"]),
        errors="coerce"
    ),
    "air_temp": weather_data["wAT"],
    "track_temp": weather_data["wTT"],
    "humidity": weather_data["wH"],
    "rainfall": weather_data["wR"],
    "wind_speed": weather_data["wWS"]
}).dropna(subset=["weather_wT"])

# Sort both datasets chronologically
lap_weather = lap_weather.sort_values("lap_wT")

weather_timeline = weather_timeline.sort_values(
    "weather_wT"
)

# Match each lap record to the nearest weather observation
weather_match = pd.merge_asof(
    lap_weather,
    weather_timeline,
    left_on="lap_wT",
    right_on="weather_wT",
    direction="nearest"
)

weather_match["time_difference"] = (
    weather_match["lap_wT"]
    - weather_match["weather_wT"]
).abs()

print("\nTime difference statistics:")
print(
    weather_match["time_difference"].describe()
)

print("\nMatching coverage:")

for tolerance in [10, 30, 60, 90]:

    matched = (
        weather_match["time_difference"]
        <= tolerance
    ).sum()

    print(
        f"Within {tolerance} seconds: "
        f"{matched}/{len(weather_match)}"
    )

print("\nSample matched records:")
print(weather_match.head(10).to_string(index=False))

# ------------------------------------------------------------
# 3. TELEMETRY FILE INVESTIGATION
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("3. TELEMETRY FILE INVESTIGATION")
print("-" * 70)

telemetry_files = sorted(
    DRIVER_DIR.glob("*_tel.json")
)

print("Selected driver:", driver_code)
print("Telemetry files:", len(telemetry_files))

print("\nSample telemetry filenames:")

for file in telemetry_files[:20]:
    print(file.name)

# ------------------------------------------------------------
# 4. IDENTIFY TELEMETRY LAP NUMBER
# ------------------------------------------------------------

telemetry_lap_match = re.search(
    r"(\d+)_tel",
    telemetry_file.stem
)

if telemetry_lap_match:

    telemetry_lap = int(
        telemetry_lap_match.group(1)
    )

    print("\nTelemetry lap number:", telemetry_lap)

    available_laps = pd.to_numeric(
        pd.Series(driver_data["lap"]),
        errors="coerce"
    )

    print(
        "Lap exists in individual lap data:",
        telemetry_lap in available_laps.values
    )

else:
    print("Could not identify telemetry lap number.")

# ------------------------------------------------------------
# 5. TELEMETRY QUALITY CHECK
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("5. TELEMETRY QUALITY CHECK")
print("-" * 70)

tel = telemetry_data["tel"]

for field in ["time", "distance", "rel_distance"]:

    values = pd.to_numeric(
        pd.Series(tel[field]),
        errors="coerce"
    )

    print(f"\n{field}")

    print("Minimum:", values.min())
    print("Maximum:", values.max())
    print("Monotonically increasing:",
          values.is_monotonic_increasing)

for field in ["speed", "throttle", "brake", "rpm"]:

    values = pd.to_numeric(
        pd.Series(tel[field]),
        errors="coerce"
    )

    print(
        f"{field}: min={values.min():.2f}, "
        f"max={values.max():.2f}, "
        f"average={values.mean():.2f}"
    )

print("\n" + "=" * 70)
print("STEP 2.5 COMPLETED")
print("=" * 70)


FOR1SIGHT STEP 2.5

1. DATA DICTIONARY
$.files.weather.json.fields.wT: {'full_name': 'Time', 'description': 'Session time when this weather sample was recorded from the start of session', 'source': 'FastF1', 'unit': 'seconds'}
$.files.weather.json.fields.wAT: {'full_name': 'Air Temperature', 'description': 'Ambient air temperature around the track', 'source': 'FastF1', 'unit': '°C (Celsius)'}
$.files.weather.json.fields.wH: {'full_name': 'Humidity', 'description': 'Relative humidity of the air', 'source': 'FastF1', 'unit': '% (percentage)'}
$.files.weather.json.fields.wTT: {'full_name': 'Track Temperature', 'description': 'Temperature of the asphalt surface', 'source': 'FastF1', 'unit': '°C (Celsius)'}
$.files.rcm.json.fields.time: {'full_name': 'Time', 'description': 'Session time when this message was issued', 'source': 'FastF1', 'values': 'Time'}
$.files.tel.json.fields.time: {'full_name': 'Time', 'description': 'Time when this data sample was recorded from start of the lap', 'sourc

In [9]:

# ============================================================
# FOR1SIGHT - STEP 2.6
# FINAL TELEMETRY AND WEATHER VALIDATION
# ============================================================

import pandas as pd
import json
import re
from pathlib import Path

print("=" * 70)
print("FOR1SIGHT STEP 2.6 - FINAL DATA VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# A. VERIFY TELEMETRY FILES AGAINST LAP RECORDS
# ------------------------------------------------------------

print("\nA. TELEMETRY COVERAGE VALIDATION")

# Extract available lap numbers
lap_numbers = set(
    pd.to_numeric(
        pd.Series(driver_data["lap"]),
        errors="coerce"
    ).dropna().astype(int)
)

# Extract lap numbers from telemetry filenames
telemetry_mapping = {}

for file_path in DRIVER_DIR.glob("*_tel.json"):

    match = re.fullmatch(
        r"(\d+)_tel\.json",
        file_path.name.lower()
    )

    if match:

        lap_number = int(match.group(1))

        if lap_number in telemetry_mapping:
            print("WARNING: Duplicate telemetry lap:", lap_number)

        telemetry_mapping[lap_number] = file_path

telemetry_laps = set(telemetry_mapping.keys())

missing_telemetry = sorted(
    lap_numbers - telemetry_laps
)

extra_telemetry = sorted(
    telemetry_laps - lap_numbers
)

print("Lap records:", len(lap_numbers))
print("Telemetry files:", len(telemetry_laps))
print("Missing telemetry:", missing_telemetry)
print("Extra telemetry:", extra_telemetry)

if lap_numbers == telemetry_laps:
    print("RESULT: All lap numbers have matching telemetry files.")
else:
    print("RESULT: Coverage mismatch detected.")

# ------------------------------------------------------------
# B. INSPECT ALL TELEMETRY FILES
# ------------------------------------------------------------

print("\nB. TELEMETRY QUALITY ACROSS ALL LAPS")

telemetry_summary = []

for lap_number, file_path in sorted(telemetry_mapping.items()):

    try:

        with open(
            file_path,
            "r",
            encoding="utf-8-sig"
        ) as f:

            data = json.load(f)

        tel = data.get("tel", {})

        array_lengths = [
            len(value)
            for value in tel.values()
            if isinstance(value, list)
        ]

        consistent = (
            len(set(array_lengths)) <= 1
        )

        sample_count = (
            array_lengths[0]
            if array_lengths else 0
        )

        telemetry_summary.append({
            "Lap": lap_number,
            "Samples": sample_count,
            "Arrays_Consistent": consistent,
            "Time_Increasing": pd.Series(
                tel.get("time", [])
            ).is_monotonic_increasing,
            "Distance_Increasing": pd.Series(
                tel.get("distance", [])
            ).is_monotonic_increasing
        })

    except Exception as e:

        telemetry_summary.append({
            "Lap": lap_number,
            "Samples": 0,
            "Arrays_Consistent": False,
            "Time_Increasing": False,
            "Distance_Increasing": False,
            "Error": str(e)
        })

telemetry_quality_df = pd.DataFrame(
    telemetry_summary
)

display(telemetry_quality_df)

print("\nTELEMETRY SUMMARY")
print("Total samples:",
      telemetry_quality_df["Samples"].sum())

print("Consistent arrays:",
      telemetry_quality_df["Arrays_Consistent"].sum(),
      "/", len(telemetry_quality_df))

print("Increasing time:",
      telemetry_quality_df["Time_Increasing"].sum(),
      "/", len(telemetry_quality_df))

print("Increasing distance:",
      telemetry_quality_df["Distance_Increasing"].sum(),
      "/", len(telemetry_quality_df))

# ------------------------------------------------------------
# C. CHECK WEATHER TIMESTAMP DUPLICATES
# ------------------------------------------------------------

print("\nC. WEATHER TIMESTAMP VALIDATION")

weather_df = pd.DataFrame(weather_data)

lap_df = pd.DataFrame(session_data)

weather_duplicates = weather_df["wT"].duplicated().sum()

print("Weather observations:", len(weather_df))
print("Duplicate weather timestamps:", weather_duplicates)

print(
    "Unique weather timestamps:",
    weather_df["wT"].nunique()
)

# ------------------------------------------------------------
# D. COMPARE WEATHER MEASUREMENTS
# ------------------------------------------------------------

print("\nD. WEATHER VALUE COMPARISON")

weather_fields = [
    "wAT", "wTT", "wH",
    "wP", "wR", "wWD", "wWS"
]

# Only compare fields present in both datasets
common_fields = [
    field for field in weather_fields
    if field in weather_df.columns
    and field in lap_df.columns
]

if weather_duplicates == 0:

    exact_weather_match = lap_df.merge(
        weather_df,
        on="wT",
        how="left",
        validate="many_to_one",
        indicator=True,
        suffixes=("_lap", "_weather")
    )

    print(
        "Matched timestamps:",
        (exact_weather_match["_merge"] == "both").sum()
    )

    print(
        "Unmatched timestamps:",
        (exact_weather_match["_merge"] == "left_only").sum()
    )

    for field in common_fields:

        left = exact_weather_match[
            f"{field}_lap"
        ]

        right = exact_weather_match[
            f"{field}_weather"
        ]

        numeric_left = pd.to_numeric(
            left, errors="coerce"
        )

        numeric_right = pd.to_numeric(
            right, errors="coerce"
        )

        if (
            numeric_left.notna().sum()
            == left.notna().sum()
            and numeric_right.notna().sum()
            == right.notna().sum()
        ):

            equal = (
                numeric_left.eq(numeric_right)
                | (
                    numeric_left.isna()
                    & numeric_right.isna()
                )
            )

        else:

            equal = (
                left.eq(right)
                | (left.isna() & right.isna())
            )

        print(
            f"{field:<10} "
            f"Mismatches: {(~equal).sum()}"
        )

else:

    print(
        "Duplicate weather timestamps exist. "
        "Skipping many-to-one join."
    )

print("\n" + "=" * 70)
print("STEP 2.6 COMPLETED")
print("=" * 70)


FOR1SIGHT STEP 2.6 - FINAL DATA VALIDATION

A. TELEMETRY COVERAGE VALIDATION
Lap records: 24
Telemetry files: 24
Missing telemetry: []
Extra telemetry: []
RESULT: All lap numbers have matching telemetry files.

B. TELEMETRY QUALITY ACROSS ALL LAPS


,Lap,Samples,Arrays_Consistent,Time_Increasing,Distance_Increasing
0,1,1029,True,True,True
1,2,956,True,True,True
2,3,910,True,True,True
3,4,651,True,True,True
4,5,656,True,True,True
5,6,800,True,True,True
6,7,593,True,True,True
7,8,842,True,True,True
8,9,647,True,True,True
9,10,914,True,True,True



TELEMETRY SUMMARY
Total samples: 29393
Consistent arrays: 24 / 24
Increasing time: 24 / 24
Increasing distance: 22 / 24

C. WEATHER TIMESTAMP VALIDATION
Weather observations: 82
Duplicate weather timestamps: 0
Unique weather timestamps: 82

D. WEATHER VALUE COMPARISON
Matched timestamps: 487
Unmatched timestamps: 0
wAT        Mismatches: 0
wTT        Mismatches: 0
wH         Mismatches: 0
wP         Mismatches: 0
wR         Mismatches: 0
wWD        Mismatches: 0
wWS        Mismatches: 0

STEP 2.6 COMPLETED


In [10]:

# ============================================================
# FOR1SIGHT - STEP 2.7
# INVESTIGATE TELEMETRY DISTANCE ANOMALIES
# ============================================================

import json
import pandas as pd
import numpy as np

print("=" * 70)
print("FOR1SIGHT TELEMETRY ANOMALY INVESTIGATION")
print("=" * 70)

PROBLEM_LAPS = [18, 24]

for lap_number in PROBLEM_LAPS:

    print("\n" + "=" * 70)
    print(f"INVESTIGATING LAP {lap_number}")
    print("=" * 70)

    # Load telemetry
    telemetry_path = DRIVER_DIR / f"{lap_number}_tel.json"

    with open(telemetry_path, "r", encoding="utf-8-sig") as f:
        raw = json.load(f)

    tel = raw["tel"]

    # Convert telemetry into a DataFrame
    df = pd.DataFrame(tel)

    # Ensure numerical values
    for column in [
        "time", "distance", "rel_distance",
        "speed", "throttle", "brake", "rpm"
    ]:

        if column in df.columns:
            df[column] = pd.to_numeric(
                df[column], errors="coerce"
            )

    # Calculate distance differences
    df["distance_diff"] = df["distance"].diff()

    # Identify decreases and repeated distances
    decreases = df[df["distance_diff"] < 0]
    repeated = df[df["distance_diff"] == 0]

    print("\nBASIC INFORMATION")
    print("Total samples:", len(df))
    print("Distance decreases:", len(decreases))
    print("Repeated distances:", len(repeated))
    print("Minimum distance:", df["distance"].min())
    print("Maximum distance:", df["distance"].max())
    print("Minimum distance change:",
          df["distance_diff"].min())

    # Inspect records around anomalies
    print("\nANOMALY DETAILS")

    if len(decreases) == 0:

        print("No decreasing distance values.")

    else:

        for index in decreases.index[:15]:

            start = max(0, index - 2)
            end = min(len(df), index + 3)

            print(f"\nAnomaly around sample index {index}")

            display(
                df.loc[
                    start:end-1,
                    [
                        "time",
                        "distance",
                        "distance_diff",
                        "rel_distance",
                        "speed",
                        "throttle",
                        "brake",
                        "gear"
                    ]
                ]
            )

    # Check relative distance
    print("\nRELATIVE DISTANCE VALIDATION")

    rel_diff = df["rel_distance"].diff()

    print("Minimum relative distance change:",
          rel_diff.min())

    print("Relative distance decreases:",
          (rel_diff < 0).sum())

    # Check duplicate records
    print("\nDUPLICATE SAMPLE VALIDATION")

    duplicate_count = df.duplicated(
        subset=["time", "distance"]
    ).sum()

    print("Duplicate time-distance records:",
          duplicate_count)

    # Check time progression
    print("\nTIME VALIDATION")

    print("Time increasing:",
          df["time"].is_monotonic_increasing)

    print("Time duplicates:",
          df["time"].duplicated().sum())

print("\n" + "=" * 70)
print("STEP 2.7 COMPLETED")
print("=" * 70)


FOR1SIGHT TELEMETRY ANOMALY INVESTIGATION

INVESTIGATING LAP 18

BASIC INFORMATION
Total samples: 3578
Distance decreases: 742
Repeated distances: 1197
Minimum distance: 2.9440197566010795e-05
Maximum distance: 5223.887632825075
Minimum distance change: -0.02390876304525591

ANOMALY DETAILS

Anomaly around sample index 815


,time,distance,distance_diff,rel_distance,speed,throttle,brake,gear
813,109.702,5106.944167,0.056598,0.977080,7.000000,0.0,1,1
814,109.837,5106.968075,0.023909,0.977084,3.062498,0.0,1,1
815,109.942,5106.944167,-0.023909,0.977080,0.000000,0.0,1,1
816,110.017,5106.935014,-0.009153,0.977078,0.000000,0.0,1,1
817,110.342,5106.944167,0.009153,0.977080,0.000000,0.0,1,1



Anomaly around sample index 816


,time,distance,distance_diff,rel_distance,speed,throttle,brake,gear
814,109.837,5106.968075,0.023909,0.977084,3.062498,0.0,1,1
815,109.942,5106.944167,-0.023909,0.977080,0.000000,0.0,1,1
816,110.017,5106.935014,-0.009153,0.977078,0.000000,0.0,1,1
817,110.342,5106.944167,0.009153,0.977080,0.000000,0.0,1,1
818,110.367,5106.944819,0.000652,0.977080,0.000000,0.0,1,1



Anomaly around sample index 819


,time,distance,distance_diff,rel_distance,speed,throttle,brake,gear
817,110.342,5106.944167,0.009153,0.97708,0.0,0.0,1,1
818,110.367,5106.944819,0.000652,0.97708,0.0,0.0,1,1
819,110.742,5106.944167,-0.000652,0.97708,0.0,0.0,1,1
820,110.827,5106.943812,-0.000354,0.97708,0.0,0.0,1,1
821,111.107,5106.943995,0.000182,0.97708,0.0,0.0,1,1


... (51 outputs left out)

,time,distance,distance_diff,rel_distance,speed,throttle,brake,gear
631,84.797,3589.286667,0.000000e+00,1.0,0.0,0.0,0,8
632,84.964,3589.286667,0.000000e+00,1.0,0.0,0.0,0,8
633,85.147,3589.286667,-4.547474e-13,1.0,0.0,0.0,0,8
634,85.284,3589.286667,4.547474e-13,1.0,0.0,0.0,0,8
635,85.397,3589.286667,0.000000e+00,1.0,0.0,0.0,0,8



Anomaly around sample index 641


,time,distance,distance_diff,rel_distance,speed,throttle,brake,gear
639,85.884,3589.286667,0.000000e+00,1.0,0.0,0.0,0,8
640,86.127,3589.286667,9.094947e-13,1.0,0.0,0.0,0,8
641,86.284,3589.286667,-9.094947e-13,1.0,0.0,0.0,0,8
642,86.427,3589.286667,0.000000e+00,1.0,0.0,0.0,0,8
643,86.617,3589.286667,4.547474e-13,1.0,0.0,0.0,0,8



RELATIVE DISTANCE VALIDATION
Minimum relative distance change: -2.4287377313747527e-06
Relative distance decreases: 1686

DUPLICATE SAMPLE VALIDATION
Duplicate time-distance records: 0

TIME VALIDATION
Time increasing: True
Time duplicates: 0

STEP 2.7 COMPLETED


In [11]:

# ============================================================
# FOR1SIGHT - STEP 2.8
# TOLERANCE-BASED TELEMETRY VALIDATION
# ============================================================

import json
import pandas as pd

PROBLEM_LAPS = [18, 24]

DISTANCE_TOLERANCE = 0.05  # metres

print("=" * 70)
print("TOLERANCE-BASED TELEMETRY VALIDATION")
print("=" * 70)

validation_results = []

for lap_number in PROBLEM_LAPS:

    telemetry_path = DRIVER_DIR / f"{lap_number}_tel.json"

    with open(telemetry_path, "r", encoding="utf-8-sig") as f:
        raw = json.load(f)

    df = pd.DataFrame(raw["tel"])

    df["distance"] = pd.to_numeric(
        df["distance"], errors="coerce"
    )

    df["time"] = pd.to_numeric(
        df["time"], errors="coerce"
    )

    df["distance_diff"] = df["distance"].diff()

    # Strict decreases
    negative_changes = df[
        df["distance_diff"] < 0
    ]

    # Significant decreases beyond tolerance
    significant_decreases = df[
        df["distance_diff"] < -DISTANCE_TOLERANCE
    ]

    # Small fluctuations within tolerance
    minor_fluctuations = df[
        (df["distance_diff"] < 0) &
        (df["distance_diff"] >= -DISTANCE_TOLERANCE)
    ]

    # Time validation
    time_increasing = df["time"].is_monotonic_increasing

    result = {
        "lap": lap_number,
        "total_samples": len(df),
        "negative_changes": len(negative_changes),
        "minor_fluctuations": len(minor_fluctuations),
        "significant_decreases": len(significant_decreases),
        "time_increasing": time_increasing,
        "validation_status": (
            "PASS"
            if time_increasing and
            len(significant_decreases) == 0
            else "INVESTIGATE"
        )
    }

    validation_results.append(result)

    print(f"\nLAP {lap_number}")
    print("-" * 50)

    for key, value in result.items():
        print(f"{key}: {value}")

    if len(significant_decreases) > 0:

        print("\nSignificant anomalies:")
        display(significant_decreases.head(10))

# Final comparison
validation_df = pd.DataFrame(validation_results)

print("\n" + "=" * 70)
print("FINAL VALIDATION SUMMARY")
print("=" * 70)

display(validation_df)

print("STEP 2.8 COMPLETED")


TOLERANCE-BASED TELEMETRY VALIDATION

LAP 18
--------------------------------------------------
lap: 18
total_samples: 3578
negative_changes: 742
minor_fluctuations: 742
significant_decreases: 0
time_increasing: True
validation_status: PASS

LAP 24
--------------------------------------------------
lap: 24
total_samples: 8893
negative_changes: 2134
minor_fluctuations: 2134
significant_decreases: 0
time_increasing: True
validation_status: PASS

FINAL VALIDATION SUMMARY


,lap,total_samples,negative_changes,minor_fluctuations,significant_decreases,time_increasing,validation_status
0,18,3578,742,742,0,True,PASS
1,24,8893,2134,2134,0,True,PASS


STEP 2.8 COMPLETED


In [12]:

# ============================================================
# FOR1SIGHT - STEP 2.9A
# THREE-SEASON DATASET COVERAGE AUDIT
#
# Seasons: 2024, 2025, 2026
# Purpose: Audit dataset coverage before PySpark ETL
# ============================================================

import json
import re
import pandas as pd

from pathlib import Path
from collections import defaultdict

print("=" * 75)
print("FOR1SIGHT THREE-SEASON DATASET COVERAGE AUDIT")
print("=" * 75)

# ------------------------------------------------------------
# 1. LOCATE PROJECT AND RAW DATA
# ------------------------------------------------------------

possible_roots = [
    Path.cwd() / "raw_hf_data",
    Path.cwd().parent / "raw_hf_data"
]

RAW_ROOT = next(
    (p for p in possible_roots if p.exists()),
    None
)

if RAW_ROOT is None:
    raise FileNotFoundError(
        "Cannot locate raw_hf_data. "
        "Please check your notebook working directory."
    )

print("\nRAW DATA LOCATION:")
print(RAW_ROOT)

TARGET_SEASONS = ["2024", "2025", "2026"]

# ------------------------------------------------------------
# 2. DISCOVER DATASET FILES
# ------------------------------------------------------------

print("\n" + "-" * 75)
print("1. DISCOVERING DATASET FILES")
print("-" * 75)

session_lap_files = list(
    RAW_ROOT.rglob("session_laptimes.json")
)

individual_lap_files = list(
    RAW_ROOT.rglob("laptimes.json")
)

telemetry_by_driver = defaultdict(list)

for file_path in RAW_ROOT.rglob("*_tel.json"):

    # Exclude files outside the three target seasons
    relative = file_path.relative_to(RAW_ROOT)

    if relative.parts[0] not in TARGET_SEASONS:
        continue

    telemetry_by_driver[file_path.parent].append(
        file_path
    )

print("Session lap files:", len(session_lap_files))
print("Individual lap files:", len(individual_lap_files))

print(
    "Telemetry files:",
    sum(len(v) for v in telemetry_by_driver.values())
)

# ------------------------------------------------------------
# 3. CREATE SESSION DIRECTORY INDEX
# ------------------------------------------------------------

session_directories = set()

for file_path in session_lap_files:
    session_directories.add(file_path.parent)

for file_path in individual_lap_files:
    session_directories.add(file_path.parent.parent)

for driver_dir in telemetry_by_driver:
    session_directories.add(driver_dir.parent)

session_file_map = {
    p.parent: p for p in session_lap_files
}

# Map individual lap files to driver folders
individual_file_map = {
    p.parent: p for p in individual_lap_files
}

# ------------------------------------------------------------
# 4. HELPER FUNCTIONS
# ------------------------------------------------------------

def safe_load_json(file_path):

    try:
        with open(
            file_path,
            "r",
            encoding="utf-8-sig"
        ) as f:

            return json.load(f)

    except Exception as error:

        return {
            "__read_error__": str(error)
        }


def extract_lap_numbers(values):

    result = set()

    for value in values:

        try:

            number = float(value)

            if pd.notna(number) and number.is_integer():
                result.add(int(number))

        except (TypeError, ValueError):
            continue

    return result


def extract_telemetry_laps(file_paths):

    result = set()

    for file_path in file_paths:

        match = re.fullmatch(
            r"(\d+)_tel\.json",
            file_path.name.lower()
        )

        if match:
            result.add(int(match.group(1)))

    return result


# ------------------------------------------------------------
# 5. AUDIT EACH SESSION AND DRIVER
# ------------------------------------------------------------

print("\n" + "-" * 75)
print("2. AUDITING SESSIONS AND DRIVER COVERAGE")
print("-" * 75)

session_reports = []
driver_reports = []

for session_dir in sorted(session_directories):

    relative = session_dir.relative_to(RAW_ROOT)

    if len(relative.parts) < 3:
        continue

    season = relative.parts[0]

    if season not in TARGET_SEASONS:
        continue

    event_name = relative.parts[1]
    session_name = relative.parts[2]

    # Load consolidated session data if available
    session_path = session_file_map.get(session_dir)

    session_data = (
        safe_load_json(session_path)
        if session_path else None
    )

    session_driver_laps = defaultdict(set)

    session_record_count = 0
    session_schema_valid = True

    if (
        isinstance(session_data, dict)
        and "__read_error__" not in session_data
    ):

        drivers = session_data.get("drv", [])
        laps = session_data.get("lap", [])

        session_record_count = len(laps)

        if len(drivers) != len(laps):
            session_schema_valid = False

        for driver, lap in zip(drivers, laps):

            if driver is None:
                continue

            valid_laps = extract_lap_numbers([lap])

            if valid_laps:
                session_driver_laps[
                    str(driver).upper()
                ].update(valid_laps)

    # Identify driver directories
    driver_dirs = {
        p for p in individual_file_map
        if p.parent == session_dir
    }

    driver_dirs.update({
        p for p in telemetry_by_driver
        if p.parent == session_dir
    })

    # Add drivers discovered in session-level data
    for driver_code in session_driver_laps:

        candidate = session_dir / driver_code

        if candidate.is_dir():
            driver_dirs.add(candidate)

    session_driver_count = 0
    session_individual_records = 0
    session_telemetry_count = 0
    session_missing_telemetry = 0
    session_extra_telemetry = 0
    session_lap_mismatches = 0

    for driver_dir in sorted(driver_dirs):

        driver_code = driver_dir.name.upper()

        individual_path = individual_file_map.get(
            driver_dir
        )

        individual_laps = set()

        individual_record_count = 0

        if individual_path:

            individual_data = safe_load_json(
                individual_path
            )

            if (
                isinstance(individual_data, dict)
                and "__read_error__" not in individual_data
            ):

                individual_laps_list = individual_data.get(
                    "lap", []
                )

                individual_record_count = len(
                    individual_laps_list
                )

                individual_laps = extract_lap_numbers(
                    individual_laps_list
                )

        # Get session-level laps for this driver
        consolidated_laps = session_driver_laps.get(
            driver_code, set()
        )

        # Use individual lap data as primary coverage reference.
        # If unavailable, use session-level records.
        expected_laps = (
            individual_laps
            if individual_path
            else consolidated_laps
        )

        # Telemetry files for this driver
        telemetry_files = telemetry_by_driver.get(
            driver_dir, []
        )

        telemetry_laps = extract_telemetry_laps(
            telemetry_files
        )

        missing_telemetry = (
            expected_laps - telemetry_laps
        )

        extra_telemetry = (
            telemetry_laps - expected_laps
        )

        # Compare individual and session-level lap numbers
        session_only = (
            consolidated_laps - individual_laps
        )

        individual_only = (
            individual_laps - consolidated_laps
        )

        if individual_path and consolidated_laps:

            mismatch_count = (
                len(session_only) + len(individual_only)
            )

        else:
            mismatch_count = 0

        driver_reports.append({

            "Season": season,
            "Event": event_name,
            "Session": session_name,
            "Driver": driver_code,

            "Individual_Lap_Records":
                individual_record_count,

            "Session_Lap_Count":
                len(consolidated_laps),

            "Telemetry_Files":
                len(telemetry_files),

            "Expected_Laps":
                len(expected_laps),

            "Missing_Telemetry":
                len(missing_telemetry),

            "Extra_Telemetry":
                len(extra_telemetry),

            "Session_Individual_Lap_Mismatches":
                mismatch_count,

            "Missing_Telemetry_Lap_Numbers":
                sorted(missing_telemetry),

            "Extra_Telemetry_Lap_Numbers":
                sorted(extra_telemetry)
        })

        session_driver_count += 1

        session_individual_records += (
            individual_record_count
        )

        session_telemetry_count += len(
            telemetry_files
        )

        session_missing_telemetry += len(
            missing_telemetry
        )

        session_extra_telemetry += len(
            extra_telemetry
        )

        session_lap_mismatches += mismatch_count

    session_reports.append({

        "Season": season,
        "Event": event_name,
        "Session": session_name,

        "Has_Session_Lap_File":
            session_path is not None,

        "Session_Lap_Records":
            session_record_count,

        "Session_Schema_Valid":
            session_schema_valid,

        "Driver_Count":
            session_driver_count,

        "Individual_Lap_Records":
            session_individual_records,

        "Telemetry_Files":
            session_telemetry_count,

        "Missing_Telemetry":
            session_missing_telemetry,

        "Extra_Telemetry":
            session_extra_telemetry,

        "Lap_Record_Mismatches":
            session_lap_mismatches
    })

    if len(session_reports) % 25 == 0:

        print(
            "Sessions audited:",
            len(session_reports)
        )

print("\nTotal sessions audited:", len(session_reports))

# ------------------------------------------------------------
# 6. CONVERT RESULTS INTO DATAFRAMES
# ------------------------------------------------------------

session_df = pd.DataFrame(session_reports)

driver_df = pd.DataFrame(driver_reports)

if session_df.empty:
    raise RuntimeError(
        "No sessions discovered. Check the folder structure."
    )

# ------------------------------------------------------------
# 7. GENERATE SEASON SUMMARY
# ------------------------------------------------------------

print("\n" + "-" * 75)
print("3. THREE-SEASON SUMMARY")
print("-" * 75)

season_summary = (
    session_df.groupby("Season")
    .agg(

        Total_Sessions=("Session", "count"),

        Total_Drivers=("Driver_Count", "sum"),

        Session_Lap_Records=(
            "Session_Lap_Records", "sum"
        ),

        Individual_Lap_Records=(
            "Individual_Lap_Records", "sum"
        ),

        Total_Telemetry_Files=(
            "Telemetry_Files", "sum"
        ),

        Missing_Telemetry=(
            "Missing_Telemetry", "sum"
        ),

        Extra_Telemetry=(
            "Extra_Telemetry", "sum"
        ),

        Lap_Record_Mismatches=(
            "Lap_Record_Mismatches", "sum"
        ),

        Sessions_Without_Consolidated_Laps=(
            "Has_Session_Lap_File",
            lambda x: (~x).sum()
        )
    )
    .reset_index()
)

display(season_summary)

# ------------------------------------------------------------
# 8. GENERATE SESSION COVERAGE REPORT
# ------------------------------------------------------------

print("\n" + "-" * 75)
print("4. SESSION COVERAGE REPORT")
print("-" * 75)

display(
    session_df.sort_values(
        ["Season", "Event", "Session"]
    ).head(30)
)

# ------------------------------------------------------------
# 9. GENERATE DRIVER COVERAGE REPORT
# ------------------------------------------------------------

print("\n" + "-" * 75)
print("5. DRIVER COVERAGE REPORT")
print("-" * 75)

display(
    driver_df.head(30)
)

# ------------------------------------------------------------
# 10. IDENTIFY PROBLEMATIC RECORDS
# ------------------------------------------------------------

print("\n" + "-" * 75)
print("6. DATA COVERAGE WARNINGS")
print("-" * 75)

if not driver_df.empty:

    missing_report = driver_df[
        driver_df["Missing_Telemetry"] > 0
    ]

    extra_report = driver_df[
        driver_df["Extra_Telemetry"] > 0
    ]

    mismatch_report = driver_df[
        driver_df[
            "Session_Individual_Lap_Mismatches"
        ] > 0
    ]

    print(
        "Drivers with missing telemetry:",
        len(missing_report)
    )

    print(
        "Drivers with extra telemetry:",
        len(extra_report)
    )

    print(
        "Drivers with lap record mismatches:",
        len(mismatch_report)
    )

    if not missing_report.empty:
        display(missing_report.head(20))

    if not extra_report.empty:
        display(extra_report.head(20))

    if not mismatch_report.empty:
        display(mismatch_report.head(20))

# ------------------------------------------------------------
# 11. SAVE AUDIT REPORTS
# ------------------------------------------------------------

print("\n" + "-" * 75)
print("7. SAVING AUDIT REPORTS")
print("-" * 75)

REPORT_DIR = RAW_ROOT.parent / "audit_reports"

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

season_summary.to_csv(
    REPORT_DIR / "step2_9_season_summary.csv",
    index=False
)

session_df.to_csv(
    REPORT_DIR / "step2_9_session_coverage.csv",
    index=False
)

driver_df.to_csv(
    REPORT_DIR / "step2_9_driver_coverage.csv",
    index=False
)

print("Report directory:", REPORT_DIR)

print("\nGenerated files:")

for report in REPORT_DIR.glob("step2_9_*.csv"):
    print(" -", report.name)

print("\n" + "=" * 75)
print("STEP 2.9A COMPLETED")
print("=" * 75)


FOR1SIGHT THREE-SEASON DATASET COVERAGE AUDIT

RAW DATA LOCATION:
c:\Users\ayepd\Desktop\For1Sight Project\raw_hf_data

---------------------------------------------------------------------------
1. DISCOVERING DATASET FILES
---------------------------------------------------------------------------
Session lap files: 325
Individual lap files: 6904
Telemetry files: 197145

---------------------------------------------------------------------------
2. AUDITING SESSIONS AND DRIVER COVERAGE
---------------------------------------------------------------------------
Sessions audited: 25
Sessions audited: 50
Sessions audited: 75
Sessions audited: 100
Sessions audited: 125
Sessions audited: 150
Sessions audited: 175
Sessions audited: 200
Sessions audited: 225
Sessions audited: 250
Sessions audited: 275
Sessions audited: 300
Sessions audited: 325

Total sessions audited: 340

---------------------------------------------------------------------------
3. THREE-SEASON SUMMARY
------------------

,Season,Total_Sessions,Total_Drivers,Session_Lap_Records,Individual_Lap_Records,Total_Telemetry_Files,Missing_Telemetry,Extra_Telemetry,Lap_Record_Mismatches,Sessions_Without_Consolidated_Laps
0,2024,123,2440,64043,67738,67616,122,0,0,3
1,2025,123,2450,65655,69528,69459,69,0,0,3
2,2026,94,2014,49546,60147,60070,103,26,0,9



---------------------------------------------------------------------------
4. SESSION COVERAGE REPORT
---------------------------------------------------------------------------


,Season,Event,Session,Has_Session_Lap_File,Session_Lap_Records,Session_Schema_Valid,Driver_Count,Individual_Lap_Records,Telemetry_Files,Missing_Telemetry,Extra_Telemetry,Lap_Record_Mismatches
0,2024,Abu Dhabi Grand Prix,Practice 1,True,471,True,20,471,471,0,0,0
1,2024,Abu Dhabi Grand Prix,Practice 2,True,507,True,20,507,507,0,0,0
2,2024,Abu Dhabi Grand Prix,Practice 3,True,342,True,20,342,342,0,0,0
3,2024,Abu Dhabi Grand Prix,Qualifying,True,261,True,20,261,261,0,0,0
4,2024,Abu Dhabi Grand Prix,Race,True,1035,True,20,1035,1035,0,0,0
5,2024,Australian Grand Prix,Practice 1,True,436,True,20,436,434,2,0,0
6,2024,Australian Grand Prix,Practice 2,True,532,True,19,532,532,0,0,0
7,2024,Australian Grand Prix,Practice 3,True,397,True,19,397,397,0,0,0
8,2024,Australian Grand Prix,Qualifying,True,310,True,19,310,310,0,0,0
9,2024,Australian Grand Prix,Race,True,998,True,19,998,998,0,0,0



---------------------------------------------------------------------------
5. DRIVER COVERAGE REPORT
---------------------------------------------------------------------------


,Season,Event,Session,Driver,Individual_Lap_Records,Session_Lap_Count,Telemetry_Files,Expected_Laps,Missing_Telemetry,Extra_Telemetry,Session_Individual_Lap_Mismatches,Missing_Telemetry_Lap_Numbers,Extra_Telemetry_Lap_Numbers
0,2024,Abu Dhabi Grand Prix,Practice 1,ALO,23,23,23,23,0,0,0,[],[]
1,2024,Abu Dhabi Grand Prix,Practice 1,BOT,23,23,23,23,0,0,0,[],[]
2,2024,Abu Dhabi Grand Prix,Practice 1,BRO,22,22,22,22,0,0,0,[],[]
3,2024,Abu Dhabi Grand Prix,Practice 1,COL,24,24,24,24,0,0,0,[],[]
4,2024,Abu Dhabi Grand Prix,Practice 1,DOO,24,24,24,24,0,0,0,[],[]
5,2024,Abu Dhabi Grand Prix,Practice 1,DRU,22,22,22,22,0,0,0,[],[]
6,2024,Abu Dhabi Grand Prix,Practice 1,GAS,24,24,24,24,0,0,0,[],[]
7,2024,Abu Dhabi Grand Prix,Practice 1,HAD,22,22,22,22,0,0,0,[],[]
8,2024,Abu Dhabi Grand Prix,Practice 1,HAM,27,27,27,27,0,0,0,[],[]
9,2024,Abu Dhabi Grand Prix,Practice 1,HIR,19,19,19,19,0,0,0,[],[]



---------------------------------------------------------------------------
6. DATA COVERAGE WARNINGS
---------------------------------------------------------------------------
Drivers with missing telemetry: 143
Drivers with extra telemetry: 26
Drivers with lap record mismatches: 0


,Season,Event,Session,Driver,Individual_Lap_Records,Session_Lap_Count,Telemetry_Files,Expected_Laps,Missing_Telemetry,Extra_Telemetry,Session_Individual_Lap_Mismatches,Missing_Telemetry_Lap_Numbers,Extra_Telemetry_Lap_Numbers
114,2024,Australian Grand Prix,Practice 1,SAI,21,21,19,21,2,0,0,"[1, 2]",[]
196,2024,Austrian Grand Prix,Practice 1,ALB,28,28,27,28,1,0,0,[1],[]
197,2024,Austrian Grand Prix,Practice 1,ALO,30,30,25,30,5,0,0,"[1, 2, 3, 4, 5]",[]
198,2024,Austrian Grand Prix,Practice 1,BOT,29,29,25,29,4,0,0,"[1, 2, 3, 4]",[]
199,2024,Austrian Grand Prix,Practice 1,GAS,28,28,23,28,5,0,0,"[1, 2, 3, 4, 5]",[]
200,2024,Austrian Grand Prix,Practice 1,HAM,39,39,34,39,5,0,0,"[1, 2, 3, 4, 5]",[]
201,2024,Austrian Grand Prix,Practice 1,HUL,31,31,26,31,5,0,0,"[1, 2, 3, 4, 5]",[]
202,2024,Austrian Grand Prix,Practice 1,LEC,31,31,26,31,5,0,0,"[1, 2, 3, 4, 5]",[]
203,2024,Austrian Grand Prix,Practice 1,MAG,31,31,27,31,4,0,0,"[1, 2, 3, 4]",[]
204,2024,Austrian Grand Prix,Practice 1,NOR,30,30,25,30,5,0,0,"[1, 2, 3, 4, 5]",[]


,Season,Event,Session,Driver,Individual_Lap_Records,Session_Lap_Count,Telemetry_Files,Expected_Laps,Missing_Telemetry,Extra_Telemetry,Session_Individual_Lap_Mismatches,Missing_Telemetry_Lap_Numbers,Extra_Telemetry_Lap_Numbers
4979,2026,Australian Grand Prix,Race,BOT,17,17,17,17,1,1,0,[17],[18]
5102,2026,Austrian Grand Prix,Race,SAI,25,25,26,25,0,1,0,[],[26]
5115,2026,Azerbaijan Grand Prix,Practice 1,HUL,23,23,24,23,0,1,0,[],[24]
5200,2026,Azerbaijan Grand Prix,Race,GAS,37,37,37,37,1,1,0,[37],[38]
5207,2026,Azerbaijan Grand Prix,Race,NOR,37,37,37,37,1,1,0,[37],[38]
5241,2026,Bahrain Grand Prix,Practice 2,BOR,26,26,26,26,1,1,0,[26],[27]
5303,2026,Bahrain Grand Prix,Race,ALB,43,43,43,43,1,1,0,[43],[44]
5308,2026,Bahrain Grand Prix,Race,BOT,9,9,9,9,1,1,0,[9],[10]
5321,2026,Bahrain Grand Prix,Race,RUS,51,51,51,51,1,1,0,[51],[52]
5413,2026,Barcelona Grand Prix,Race,ALO,39,39,39,39,1,1,0,[39],[40]



---------------------------------------------------------------------------
7. SAVING AUDIT REPORTS
---------------------------------------------------------------------------
Report directory: c:\Users\ayepd\Desktop\For1Sight Project\audit_reports

Generated files:
 - step2_9_driver_coverage.csv
 - step2_9_season_summary.csv
 - step2_9_session_coverage.csv

STEP 2.9A COMPLETED


In [13]:

# ============================================================
# FOR1SIGHT - STEP 2.9A.1
# COVERAGE ANOMALY INVESTIGATION
# ============================================================

import pandas as pd
from pathlib import Path

REPORT_DIR = RAW_ROOT.parent / "audit_reports"

driver_df = pd.read_csv(
    REPORT_DIR / "step2_9_driver_coverage.csv"
)

session_df = pd.read_csv(
    REPORT_DIR / "step2_9_session_coverage.csv"
)

print("=" * 75)
print("FOR1SIGHT COVERAGE ANOMALY INVESTIGATION")
print("=" * 75)


# ------------------------------------------------------------
# 1. MISSING TELEMETRY BY SEASON
# ------------------------------------------------------------

print("\n1. MISSING TELEMETRY BY SEASON")

missing = driver_df[
    driver_df["Missing_Telemetry"] > 0
]

missing_summary = (
    missing.groupby("Season")
    .agg(
        Affected_Driver_Sessions=("Driver", "count"),
        Missing_Laps=("Missing_Telemetry", "sum")
    )
    .reset_index()
)

display(missing_summary)


# ------------------------------------------------------------
# 2. MISSING TELEMETRY BY GRAND PRIX
# ------------------------------------------------------------

print("\n2. GRAND PRIX WITH MOST MISSING TELEMETRY")

missing_gp = (
    missing.groupby(
        ["Season", "Event", "Session"]
    )
    .agg(
        Affected_Drivers=("Driver", "count"),
        Missing_Laps=("Missing_Telemetry", "sum")
    )
    .reset_index()
    .sort_values(
        "Missing_Laps",
        ascending=False
    )
)

display(missing_gp.head(25))


# ------------------------------------------------------------
# 3. EXTRA TELEMETRY INVESTIGATION
# ------------------------------------------------------------

print("\n3. EXTRA TELEMETRY BY SEASON")

extra = driver_df[
    driver_df["Extra_Telemetry"] > 0
]

extra_summary = (
    extra.groupby("Season")
    .agg(
        Affected_Driver_Sessions=("Driver", "count"),
        Extra_Laps=("Extra_Telemetry", "sum")
    )
    .reset_index()
)

display(extra_summary)

print("\nEXTRA TELEMETRY DETAILS")

display(
    extra[
        [
            "Season",
            "Event",
            "Session",
            "Driver",
            "Expected_Laps",
            "Telemetry_Files",
            "Extra_Telemetry",
            "Extra_Telemetry_Lap_Numbers"
        ]
    ].head(40)
)


# ------------------------------------------------------------
# 4. MISSING CONSOLIDATED SESSION FILES
# ------------------------------------------------------------

print("\n4. MISSING SESSION LAP-TIME FILES")

missing_sessions = session_df[
    ~session_df["Has_Session_Lap_File"]
]

display(
    missing_sessions[
        [
            "Season",
            "Event",
            "Session",
            "Driver_Count",
            "Individual_Lap_Records",
            "Telemetry_Files"
        ]
    ]
)


# ------------------------------------------------------------
# 5. RECONCILE SESSION VS DRIVER RECORD COUNTS
# ------------------------------------------------------------

print("\n5. SESSION RECORD COUNT RECONCILIATION")

grouped = (
    driver_df.groupby(
        ["Season", "Event", "Session"]
    )
    .agg(
        Driver_Entries=("Driver", "count"),

        Individual_Raw_Rows=(
            "Individual_Lap_Records", "sum"
        ),

        Session_Unique_Laps=(
            "Session_Lap_Count", "sum"
        ),

        Telemetry_Files=(
            "Telemetry_Files", "sum"
        )
    )
    .reset_index()
)

reconciliation = session_df.merge(
    grouped,
    on=["Season", "Event", "Session"],
    how="left"
)

reconciliation["Session_Row_Gap"] = (
    reconciliation["Session_Lap_Records"]
    - reconciliation["Session_Unique_Laps"]
)

reconciliation["Individual_Raw_vs_Unique_Gap"] = (
    reconciliation["Individual_Raw_Rows"]
    - reconciliation["Session_Unique_Laps"]
)

display(
    reconciliation[
        [
            "Season",
            "Event",
            "Session",
            "Has_Session_Lap_File",
            "Session_Lap_Records",
            "Individual_Raw_Rows",
            "Session_Unique_Laps",
            "Session_Row_Gap",
            "Individual_Raw_vs_Unique_Gap"
        ]
    ].sort_values(
        "Session_Row_Gap",
        key=lambda x: x.abs(),
        ascending=False
    ).head(30)
)


# ------------------------------------------------------------
# 6. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 2.9A.1 COMPLETED")
print("=" * 75)

print("Missing telemetry entries:", len(missing))
print("Extra telemetry entries:", len(extra))
print("Missing consolidated session files:", len(missing_sessions))

print(
    "\nNo original JSON or telemetry files were modified."
)


FOR1SIGHT COVERAGE ANOMALY INVESTIGATION

1. MISSING TELEMETRY BY SEASON


,Season,Affected_Driver_Sessions,Missing_Laps
0,2024,41,122
1,2025,37,69
2,2026,65,103



2. GRAND PRIX WITH MOST MISSING TELEMETRY


,Season,Event,Session,Affected_Drivers,Missing_Laps
1,2024,Austrian Grand Prix,Practice 1,20,87
30,2025,Spanish Grand Prix,Practice 3,19,40
59,2026,Pre-Season Testing,Practice 3,9,24
61,2026,Pre-Season Testing 1,Practice 3,9,24
27,2025,Pre-Season Testing,Practice 3,6,12
13,2024,Pre-Season Testing,Practice 3,2,8
26,2025,Pre-Season Testing,Practice 2,2,7
33,2026,Austrian Grand Prix,Practice 3,1,6
37,2026,Azerbaijan Grand Prix,Race,4,4
64,2026,Singapore Grand Prix,Race,3,3



3. EXTRA TELEMETRY BY SEASON


,Season,Affected_Driver_Sessions,Extra_Laps
0,2026,26,26


... (3 outputs left out)

,Season,Event,Session,Driver_Count,Individual_Lap_Records,Telemetry_Files
90,2024,Pre-Season Testing,Practice 1,18,1236,1235
91,2024,Pre-Season Testing,Practice 2,17,1219,1217
92,2024,Pre-Season Testing,Practice 3,19,1240,1232
213,2025,Pre-Season Testing,Practice 1,20,1326,1325
214,2025,Pre-Season Testing,Practice 2,18,1257,1250
215,2025,Pre-Season Testing,Practice 3,18,1290,1278
321,2026,Pre-Season Testing,Practice 1,18,1128,1128
322,2026,Pre-Season Testing,Practice 2,16,1063,1060
323,2026,Pre-Season Testing,Practice 3,17,1265,1241
324,2026,Pre-Season Testing 1,Practice 1,18,1128,1128



5. SESSION RECORD COUNT RECONCILIATION


,Season,Event,Session,Has_Session_Lap_File,Session_Lap_Records,Individual_Raw_Rows,Session_Unique_Laps,Session_Row_Gap,Individual_Raw_vs_Unique_Gap
0,2024,Abu Dhabi Grand Prix,Practice 1,True,471,471,471,0,0
1,2024,Abu Dhabi Grand Prix,Practice 2,True,507,507,507,0,0
2,2024,Abu Dhabi Grand Prix,Practice 3,True,342,342,342,0,0
3,2024,Abu Dhabi Grand Prix,Qualifying,True,261,261,261,0,0
4,2024,Abu Dhabi Grand Prix,Race,True,1035,1035,1035,0,0
5,2024,Australian Grand Prix,Practice 1,True,436,436,436,0,0
6,2024,Australian Grand Prix,Practice 2,True,532,532,532,0,0
7,2024,Australian Grand Prix,Practice 3,True,397,397,397,0,0
8,2024,Australian Grand Prix,Qualifying,True,310,310,310,0,0
9,2024,Australian Grand Prix,Race,True,998,998,998,0,0



STEP 2.9A.1 COMPLETED
Missing telemetry entries: 143
Extra telemetry entries: 26
Missing consolidated session files: 15

No original JSON or telemetry files were modified.


In [14]:

# ============================================================
# FOR1SIGHT STEP 2.9A.2
# DETAILED ANOMALY DIAGNOSIS
# ============================================================

import pandas as pd
import ast
from pathlib import Path

REPORT_DIR = RAW_ROOT.parent / "audit_reports"

driver_df = pd.read_csv(
    REPORT_DIR / "step2_9_driver_coverage.csv"
)

session_df = pd.read_csv(
    REPORT_DIR / "step2_9_session_coverage.csv"
)

# ------------------------------------------------------------
# 1. INSPECT MISSING LAP NUMBERS
# ------------------------------------------------------------

print("=" * 70)
print("1. MISSING LAP NUMBER INVESTIGATION")
print("=" * 70)

target_sessions = [
    (2024, "Austrian Grand Prix", "Practice 1"),
    (2025, "Spanish Grand Prix", "Practice 3"),
    (2026, "Pre-Season Testing", "Practice 3"),
]

for season, event, session in target_sessions:

    subset = driver_df[
        (driver_df["Season"] == season) &
        (driver_df["Event"] == event) &
        (driver_df["Session"] == session)
    ].copy()

    print(f"\n{season} | {event} | {session}")

    columns = [
        "Driver",
        "Individual_Lap_Records",
        "Telemetry_Files",
        "Missing_Telemetry",
        "Missing_Telemetry_Lap_Numbers"
    ]

    display(subset[columns])


# ------------------------------------------------------------
# 2. INVESTIGATE EXTRA TELEMETRY PATTERN
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2. EXTRA TELEMETRY PATTERN")
print("=" * 70)

extra = driver_df[
    driver_df["Extra_Telemetry"] > 0
].copy()

def parse_lap_list(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    try:
        return ast.literal_eval(str(value))
    except (ValueError, SyntaxError):
        return []

extra["Extra_Lap_List"] = (
    extra["Extra_Telemetry_Lap_Numbers"]
    .apply(parse_lap_list)
)

extra["Expected_Max_Lap"] = extra["Expected_Laps"]

extra["Extra_Is_Expected_Plus_One"] = (
    extra.apply(
        lambda row: all(
            lap == row["Expected_Max_Lap"] + 1
            for lap in row["Extra_Lap_List"]
        ),
        axis=1
    )
)

display(
    extra[
        [
            "Season",
            "Event",
            "Session",
            "Driver",
            "Expected_Laps",
            "Extra_Lap_List",
            "Extra_Is_Expected_Plus_One"
        ]
    ]
)

print(
    "\nExtra entries following expected + 1:",
    extra["Extra_Is_Expected_Plus_One"].sum(),
    "/",
    len(extra)
)


# ------------------------------------------------------------
# 3. VERIFY 2026 PRE-SEASON SESSION COUNTS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("3. 2026 PRE-SEASON TESTING COMPARISON")
print("=" * 70)

preseason = session_df[
    (session_df["Season"] == 2026) &
    (session_df["Event"].str.contains(
        "Pre-Season Testing",
        case=False,
        na=False
    ))
]

display(preseason)


# ------------------------------------------------------------
# 4. COMPLETE SESSION RECONCILIATION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("4. COMPLETE SESSION RECONCILIATION")
print("=" * 70)

grouped = (
    driver_df.groupby(
        ["Season", "Event", "Session"]
    )
    .agg(
        Individual_Raw_Rows=(
            "Individual_Lap_Records", "sum"
        ),
        Driver_Lap_Count_Total=(
            "Session_Lap_Count", "sum"
        )
    )
    .reset_index()
)

reconciliation = session_df.merge(
    grouped,
    on=["Season", "Event", "Session"],
    how="left"
)

reconciliation["Session_Row_Gap"] = (
    reconciliation["Session_Lap_Records"].fillna(0)
    - reconciliation["Driver_Lap_Count_Total"].fillna(0)
)

reconciliation["Individual_Row_Gap"] = (
    reconciliation["Individual_Raw_Rows"].fillna(0)
    - reconciliation["Driver_Lap_Count_Total"].fillna(0)
)

problems = reconciliation[
    (reconciliation["Session_Row_Gap"] != 0) |
    (reconciliation["Individual_Row_Gap"] != 0)
]

print("Total sessions:", len(reconciliation))
print("Sessions with reconciliation differences:", len(problems))

display(
    problems[
        [
            "Season",
            "Event",
            "Session",
            "Has_Session_Lap_File",
            "Session_Lap_Records",
            "Individual_Raw_Rows",
            "Driver_Lap_Count_Total",
            "Session_Row_Gap",
            "Individual_Row_Gap"
        ]
    ]
)


# ------------------------------------------------------------
# 5. FINAL REPORT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 2.9A.2 COMPLETED")
print("=" * 70)

print("Missing telemetry entries:", 
      driver_df["Missing_Telemetry"].sum())

print("Extra telemetry entries:",
      driver_df["Extra_Telemetry"].sum())

print("Sessions requiring reconciliation:", len(problems))

print("Original data remains unchanged.")


1. MISSING LAP NUMBER INVESTIGATION

2024 | Austrian Grand Prix | Practice 1


,Driver,Individual_Lap_Records,Telemetry_Files,Missing_Telemetry,Missing_Telemetry_Lap_Numbers
196,ALB,28,27,1,[1]
197,ALO,30,25,5,"[1, 2, 3, 4, 5]"
198,BOT,29,25,4,"[1, 2, 3, 4]"
199,GAS,28,23,5,"[1, 2, 3, 4, 5]"
200,HAM,39,34,5,"[1, 2, 3, 4, 5]"
201,HUL,31,26,5,"[1, 2, 3, 4, 5]"
202,LEC,31,26,5,"[1, 2, 3, 4, 5]"
203,MAG,31,27,4,"[1, 2, 3, 4]"
204,NOR,30,25,5,"[1, 2, 3, 4, 5]"
205,OCO,30,25,5,"[1, 2, 3, 4, 5]"



2025 | Spanish Grand Prix | Practice 3


,Driver,Individual_Lap_Records,Telemetry_Files,Missing_Telemetry,Missing_Telemetry_Lap_Numbers
4632,ALB,6,4,2,"[1, 3]"
4633,ALO,16,14,2,"[1, 3]"
4634,ANT,12,12,0,[]
4635,BEA,12,11,1,[1]
4636,BOR,14,13,1,[1]
4637,COL,18,16,2,"[1, 3]"
4638,GAS,19,15,4,"[1, 2, 3, 4]"
4639,HAD,18,16,2,"[1, 3]"
4640,HAM,18,16,2,"[1, 3]"
4641,HUL,17,13,4,"[1, 2, 3, 4]"



2026 | Pre-Season Testing | Practice 3


,Driver,Individual_Lap_Records,Telemetry_Files,Missing_Telemetry,Missing_Telemetry_Lap_Numbers
6563,ALB,70,70,0,[]
6564,ANT,48,48,0,[]
6565,BEA,65,63,2,"[1, 3]"
6566,BOR,53,50,3,"[1, 3, 4]"
6567,BOT,36,33,3,"[1, 3, 7]"
6568,COL,134,131,3,"[1, 3, 4]"
6569,HAD,53,53,0,[]
6570,HAM,130,129,1,[1]
6571,HUL,48,48,0,[]
6572,LAW,108,102,6,"[1, 3, 4, 5, 75, 76]"


... (3 outputs left out)

,Season,Event,Session,Has_Session_Lap_File,Session_Lap_Records,Session_Schema_Valid,Driver_Count,Individual_Lap_Records,Telemetry_Files,Missing_Telemetry,Extra_Telemetry,Lap_Record_Mismatches
321,2026,Pre-Season Testing,Practice 1,False,0,True,18,1128,1128,0,0,0
322,2026,Pre-Season Testing,Practice 2,False,0,True,16,1063,1060,3,0,0
323,2026,Pre-Season Testing,Practice 3,False,0,True,17,1265,1241,24,0,0
324,2026,Pre-Season Testing 1,Practice 1,False,0,True,18,1128,1128,0,0,0
325,2026,Pre-Season Testing 1,Practice 2,False,0,True,16,1063,1060,3,0,0
326,2026,Pre-Season Testing 1,Practice 3,False,0,True,17,1265,1241,24,0,0
327,2026,Pre-Season Testing 2,Practice 1,False,0,True,21,1069,1068,1,0,0
328,2026,Pre-Season Testing 2,Practice 2,False,0,True,16,1278,1278,0,0,0
329,2026,Pre-Season Testing 2,Practice 3,False,0,True,17,1334,1333,1,0,0



4. COMPLETE SESSION RECONCILIATION
Total sessions: 340
Sessions with reconciliation differences: 19


,Season,Event,Session,Has_Session_Lap_File,Session_Lap_Records,Individual_Raw_Rows,Driver_Lap_Count_Total,Session_Row_Gap,Individual_Row_Gap
90,2024,Pre-Season Testing,Practice 1,False,0,1236,0,0,1236
91,2024,Pre-Season Testing,Practice 2,False,0,1219,0,0,1219
92,2024,Pre-Season Testing,Practice 3,False,0,1240,0,0,1240
213,2025,Pre-Season Testing,Practice 1,False,0,1326,0,0,1326
214,2025,Pre-Season Testing,Practice 2,False,0,1257,0,0,1257
215,2025,Pre-Season Testing,Practice 3,False,0,1290,0,0,1290
250,2026,Australian Grand Prix,Race,True,1008,1009,1008,0,1
283,2026,Canadian Grand Prix,Race,True,1214,1215,1214,0,1
288,2026,Chinese Grand Prix,Race,True,921,925,921,0,4
314,2026,Miami Grand Prix,Sprint,True,380,382,380,0,2



STEP 2.9A.2 COMPLETED
Missing telemetry entries: 294
Extra telemetry entries: 26
Sessions requiring reconciliation: 19
Original data remains unchanged.


In [15]:

# ============================================================
# FOR1SIGHT STEP 2.9A.3
# DEEP RECONCILIATION INVESTIGATION
# ============================================================

import pandas as pd

print("=" * 70)
print("FOR1SIGHT STEP 2.9A.3")
print("=" * 70)

# ------------------------------------------------------------
# 1. INVESTIGATE FOUR RECONCILIATION DIFFERENCES
# ------------------------------------------------------------

print("\n1. DRIVER-LEVEL RECONCILIATION")

target_sessions = [
    (2026, "Australian Grand Prix", "Race"),
    (2026, "Canadian Grand Prix", "Race"),
    (2026, "Chinese Grand Prix", "Race"),
    (2026, "Miami Grand Prix", "Sprint"),
]

for season, event, session in target_sessions:

    subset = driver_df[
        (driver_df["Season"] == season) &
        (driver_df["Event"] == event) &
        (driver_df["Session"] == session)
    ].copy()

    subset["Raw_Unique_Gap"] = (
        subset["Individual_Lap_Records"]
        - subset["Session_Lap_Count"]
    )

    print(f"\n{event} - {session}")

    affected = subset[
        subset["Raw_Unique_Gap"] != 0
    ]

    print("Affected drivers:", len(affected))

    display(
        affected[
            [
                "Driver",
                "Individual_Lap_Records",
                "Session_Lap_Count",
                "Raw_Unique_Gap",
                "Telemetry_Files",
                "Missing_Telemetry",
                "Extra_Telemetry"
            ]
        ]
    )


# ------------------------------------------------------------
# 2. COMPARE 2026 PRE-SEASON EVENT METADATA
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2. PRE-SEASON EVENT COMPARISON")
print("=" * 70)

base = driver_df[
    (driver_df["Season"] == 2026) &
    (driver_df["Event"] == "Pre-Season Testing")
].copy()

test1 = driver_df[
    (driver_df["Season"] == 2026) &
    (driver_df["Event"] == "Pre-Season Testing 1")
].copy()

comparison = base.merge(
    test1,
    on=["Season", "Driver", "Session"],
    how="outer",
    suffixes=("_base", "_test1"),
    indicator=True
)

print("Driver-session matching:")
print(comparison["_merge"].value_counts())

compare_columns = [
    "Individual_Lap_Records",
    "Telemetry_Files",
    "Missing_Telemetry",
    "Extra_Telemetry",
    "Session_Lap_Count"
]

for col in compare_columns:

    left = comparison[f"{col}_base"]
    right = comparison[f"{col}_test1"]

    comparison[f"{col}_equal"] = (
        left.eq(right) |
        (left.isna() & right.isna())
    )

print("\nField equality:")

for col in compare_columns:
    print(
        col,
        ":",
        comparison[f"{col}_equal"].sum(),
        "/",
        len(comparison)
    )

display(
    comparison[
        [
            "Driver",
            "Session",
            "_merge"
        ] +
        [f"{c}_equal" for c in compare_columns]
    ]
)


# ------------------------------------------------------------
# 3. INVESTIGATE EXTRA TELEMETRY LAP NUMBERS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("3. EXTRA TELEMETRY CLASSIFICATION")
print("=" * 70)

extra = driver_df[
    driver_df["Extra_Telemetry"] > 0
].copy()

print("Total extra telemetry records:", len(extra))

display(
    extra[
        [
            "Season",
            "Event",
            "Session",
            "Driver",
            "Expected_Laps",
            "Telemetry_Files",
            "Extra_Telemetry",
            "Extra_Telemetry_Lap_Numbers"
        ]
    ]
)

print("\nSeason distribution:")
print(extra.groupby("Season")["Extra_Telemetry"].sum())

print("\nSession distribution:")
print(extra.groupby("Session")["Extra_Telemetry"].sum())


# ------------------------------------------------------------
# 4. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 2.9A.3 COMPLETED")
print("=" * 70)

print("Four reconciliation sessions investigated.")
print("Pre-season metadata comparison completed.")
print("Extra telemetry entries classified for investigation.")
print("No original JSON files modified.")


FOR1SIGHT STEP 2.9A.3

1. DRIVER-LEVEL RECONCILIATION

Australian Grand Prix - Race
Affected drivers: 1


,Driver,Individual_Lap_Records,Session_Lap_Count,Raw_Unique_Gap,Telemetry_Files,Missing_Telemetry,Extra_Telemetry
4984,HUL,1,0,1,1,0,0



Canadian Grand Prix - Race
Affected drivers: 1


,Driver,Individual_Lap_Records,Session_Lap_Count,Raw_Unique_Gap,Telemetry_Files,Missing_Telemetry,Extra_Telemetry
5711,LIN,1,0,1,1,0,0



Chinese Grand Prix - Race
Affected drivers: 4


,Driver,Individual_Lap_Records,Session_Lap_Count,Raw_Unique_Gap,Telemetry_Files,Missing_Telemetry,Extra_Telemetry
5806,ALB,1,0,1,1,0,0
5810,BOR,1,0,1,1,0,0
5820,NOR,1,0,1,1,0,0
5823,PIA,1,0,1,1,0,0


... (3 outputs left out)

,Driver,Session,_merge,Individual_Lap_Records_equal,Telemetry_Files_equal,Missing_Telemetry_equal,Extra_Telemetry_equal,Session_Lap_Count_equal
0,ALB,Practice 1,both,True,True,True,True,True
1,ALB,Practice 2,both,True,True,True,True,True
2,ALB,Practice 3,both,True,True,True,True,True
3,ALO,Practice 2,both,True,True,True,True,True
4,ANT,Practice 1,both,True,True,True,True,True
5,ANT,Practice 2,both,True,True,True,True,True
6,ANT,Practice 3,both,True,True,True,True,True
7,BEA,Practice 2,both,True,True,True,True,True
8,BEA,Practice 3,both,True,True,True,True,True
9,BOR,Practice 1,both,True,True,True,True,True



3. EXTRA TELEMETRY CLASSIFICATION
Total extra telemetry records: 26


,Season,Event,Session,Driver,Expected_Laps,Telemetry_Files,Extra_Telemetry,Extra_Telemetry_Lap_Numbers
4979,2026,Australian Grand Prix,Race,BOT,17,17,1,[18]
5102,2026,Austrian Grand Prix,Race,SAI,25,26,1,[26]
5115,2026,Azerbaijan Grand Prix,Practice 1,HUL,23,24,1,[24]
5200,2026,Azerbaijan Grand Prix,Race,GAS,37,37,1,[38]
5207,2026,Azerbaijan Grand Prix,Race,NOR,37,37,1,[38]
5241,2026,Bahrain Grand Prix,Practice 2,BOR,26,26,1,[27]
5303,2026,Bahrain Grand Prix,Race,ALB,43,43,1,[44]
5308,2026,Bahrain Grand Prix,Race,BOT,9,9,1,[10]
5321,2026,Bahrain Grand Prix,Race,RUS,51,51,1,[52]
5413,2026,Barcelona Grand Prix,Race,ALO,39,39,1,[40]



Season distribution:
Season
2026    26
Name: Extra_Telemetry, dtype: int64

Session distribution:
Session
Practice 1     3
Practice 2     3
Practice 3     1
Race          19
Name: Extra_Telemetry, dtype: int64

STEP 2.9A.3 COMPLETED
Four reconciliation sessions investigated.
Pre-season metadata comparison completed.
Extra telemetry entries classified for investigation.
No original JSON files modified.


In [16]:

# ============================================================
# FOR1SIGHT STEP 2.9A.4
# RAW DATA VERIFICATION
# ============================================================

from pathlib import Path
import pandas as pd
import json
import hashlib
import re

RAW_ROOT = Path("raw_hf_data")

if not RAW_ROOT.exists():
    RAW_ROOT = Path(
        r"C:\Users\ayepd\Desktop\For1Sight Project\raw_hf_data"
    )

if not RAW_ROOT.exists():
    raise FileNotFoundError(
        f"Raw data directory not found: {RAW_ROOT}"
    )


def normalize(text):
    return re.sub(r"[^a-z0-9]", "", str(text).lower())


def find_directory(parent, name):
    if not parent.exists():
        return None

    folders = [
        p for p in parent.iterdir()
        if p.is_dir()
    ]

    target = normalize(name)

    exact = [
        p for p in folders
        if normalize(p.name) == target
    ]

    if exact:
        return exact[0]

    matches = [
        p for p in folders
        if target in normalize(p.name)
    ]

    return matches[0] if len(matches) == 1 else None


def load_json_dataframe(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, dict):
        list_fields = {
            k: pd.Series(v)
            for k, v in data.items()
            if isinstance(v, list)
        }

        if list_fields:
            return pd.DataFrame(list_fields)

    return pd.DataFrame()


def get_lap_numbers(df):
    for col in ["lap", "LapNumber", "lap_number"]:
        if col in df.columns:
            values = pd.to_numeric(
                df[col], errors="coerce"
            ).dropna()

            return sorted(values.astype(int).unique())

    return []


def get_driver_rows(df, driver):
    for col in ["drv", "driver_code", "Driver", "driver"]:
        if col in df.columns:
            return df[
                df[col].astype(str).str.upper() == driver
            ].copy()

    return None


print("=" * 70)
print("FOR1SIGHT STEP 2.9A.4")
print("RAW DATA VERIFICATION")
print("=" * 70)


# ============================================================
# 1. VERIFY THE EIGHT AFFECTED DRIVER RECORDS
# ============================================================

print("\n1. AFFECTED DRIVER RAW DATA INVESTIGATION")

targets = [
    (2026, "Australian Grand Prix", "Race", "HUL"),
    (2026, "Canadian Grand Prix", "Race", "LIN"),
    (2026, "Chinese Grand Prix", "Race", "ALB"),
    (2026, "Chinese Grand Prix", "Race", "BOR"),
    (2026, "Chinese Grand Prix", "Race", "NOR"),
    (2026, "Chinese Grand Prix", "Race", "PIA"),
    (2026, "Miami Grand Prix", "Sprint", "HUL"),
    (2026, "Miami Grand Prix", "Sprint", "LIN"),
]

for year, event, session, driver in targets:

    year_dir = RAW_ROOT / str(year)
    event_dir = find_directory(year_dir, event)

    print("\n" + "-" * 60)
    print(f"{year} | {event} | {session} | {driver}")

    if event_dir is None:
        print("EVENT DIRECTORY NOT FOUND")
        continue

    session_dir = find_directory(event_dir, session)

    if session_dir is None:
        print("SESSION DIRECTORY NOT FOUND")
        continue

    driver_dir = find_directory(session_dir, driver)

    if driver_dir is None:
        print("DRIVER DIRECTORY NOT FOUND")
        continue

    individual_file = driver_dir / "laptimes.json"

    session_file = session_dir / "session_laptimes.json"

    if not individual_file.exists():
        print("Individual laptimes.json missing")
        continue

    individual_df = load_json_dataframe(individual_file)

    individual_laps = get_lap_numbers(individual_df)

    print("Individual file:", individual_file.exists())
    print("Individual rows:", len(individual_df))
    print("Individual lap numbers:", individual_laps)

    telemetry_files = list(
        driver_dir.glob("*_tel.json")
    )

    print("Telemetry files:", len(telemetry_files))
    print(
        "Telemetry filenames:",
        sorted(f.name for f in telemetry_files)
    )

    if session_file.exists():

        session_df = load_json_dataframe(session_file)

        driver_session = get_driver_rows(
            session_df, driver
        )

        print("Session lap file: EXISTS")
        print("Session total rows:", len(session_df))

        if driver_session is not None:
            print(
                "Session records for driver:",
                len(driver_session)
            )

            print(
                "Session lap numbers:",
                get_lap_numbers(driver_session)
            )
        else:
            print("Driver identifier column not found")

    else:
        print("Session lap file: NOT FOUND")


# ============================================================
# 2. VERIFY POSSIBLE DUPLICATE PRE-SEASON DATA
# ============================================================

print("\n" + "=" * 70)
print("2. PRE-SEASON RAW FILE COMPARISON")
print("=" * 70)

year_dir = RAW_ROOT / "2026"

base_dir = find_directory(
    year_dir, "Pre-Season Testing"
)

test1_dir = find_directory(
    year_dir, "Pre-Season Testing 1"
)


def json_file_hash(path):
    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)

    canonical = json.dumps(
        obj,
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":")
    )

    return hashlib.sha256(
        canonical.encode("utf-8")
    ).hexdigest()


if base_dir and test1_dir:

    print("Dataset A:", base_dir.name)
    print("Dataset B:", test1_dir.name)

    def build_file_map(root):

        result = {}

        for file in root.rglob("*.json"):

            relative = str(
                file.relative_to(root)
            )

            result[relative] = file

        return result

    files_a = build_file_map(base_dir)
    files_b = build_file_map(test1_dir)

    common = set(files_a) & set(files_b)

    identical = 0
    different = []

    for relative in sorted(common):

        try:
            hash_a = json_file_hash(files_a[relative])
            hash_b = json_file_hash(files_b[relative])

            if hash_a == hash_b:
                identical += 1
            else:
                different.append(relative)

        except Exception as e:
            different.append(
                f"{relative} [ERROR: {e}]"
            )

    print("\nTotal JSON files A:", len(files_a))
    print("Total JSON files B:", len(files_b))
    print("Common JSON paths:", len(common))
    print("Identical JSON files:", identical)
    print("Different JSON files:", len(different))

    print(
        "Files unique to A:",
        len(set(files_a) - set(files_b))
    )

    print(
        "Files unique to B:",
        len(set(files_b) - set(files_a))
    )

    print("\nDifferent file samples:")
    print(different[:30])

else:
    print("Could not locate both pre-season directories.")
    print("Available event directories:")

    for folder in year_dir.iterdir():
        if folder.is_dir() and "preseason" in normalize(folder.name):
            print(folder.name)


# ============================================================
# 3. INVESTIGATE TELEMETRY-ONLY FINAL LAPS
# ============================================================

print("\n" + "=" * 70)
print("3. TELEMETRY-ONLY LAP INVESTIGATION")
print("=" * 70)

extra_results = []

for lap_file in year_dir.rglob("laptimes.json"):

    driver_dir = lap_file.parent

    try:
        lap_df = load_json_dataframe(lap_file)
        lap_numbers = get_lap_numbers(lap_df)

        if not lap_numbers:
            continue

        max_lap = max(lap_numbers)

        for tel_file in driver_dir.glob("*_tel.json"):

            match = re.match(
                r"(\d+)_tel\.json",
                tel_file.name
            )

            if not match:
                continue

            tel_lap = int(match.group(1))

            if tel_lap <= max_lap:
                continue

            tel_df = load_json_dataframe(tel_file)

            result = {
                "event": driver_dir.parent.parent.name,
                "session": driver_dir.parent.name,
                "driver": driver_dir.name,
                "max_individual_lap": max_lap,
                "telemetry_lap": tel_lap,
                "samples": len(tel_df)
            }

            for field in [
                "time",
                "distance",
                "rel_distance",
                "speed"
            ]:

                if field in tel_df.columns:

                    values = pd.to_numeric(
                        tel_df[field],
                        errors="coerce"
                    ).dropna()

                    if len(values):

                        result[f"{field}_min"] = values.min()
                        result[f"{field}_max"] = values.max()

                        result[f"{field}_monotonic"] = (
                            values.is_monotonic_increasing
                        )

            extra_results.append(result)

    except Exception as e:
        print("Error:", lap_file, e)


extra_df = pd.DataFrame(extra_results)

print("Telemetry-only final laps found:", len(extra_df))

if not extra_df.empty:
    display(extra_df)

    print("\nEvent distribution:")
    print(extra_df["event"].value_counts())

    print("\nDriver distribution:")
    print(extra_df["driver"].value_counts())


# ============================================================
# 4. FINAL VERIFICATION SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("STEP 2.9A.4 COMPLETED")
print("=" * 70)

print("Eight affected driver records investigated.")
print("Pre-season raw JSON comparison performed.")
print("Telemetry-only final laps inspected.")
print("Original JSON files remain unchanged.")

print("\nNO AUTOMATIC DATA CLEANING PERFORMED.")


FOR1SIGHT STEP 2.9A.4
RAW DATA VERIFICATION

1. AFFECTED DRIVER RAW DATA INVESTIGATION

------------------------------------------------------------
2026 | Australian Grand Prix | Race | HUL
Individual file: True
Individual rows: 1
Individual lap numbers: [np.int64(1)]
Telemetry files: 1
Telemetry filenames: ['1_tel.json']
Session lap file: EXISTS
Session total rows: 1008
Session records for driver: 0
Session lap numbers: []

------------------------------------------------------------
2026 | Canadian Grand Prix | Race | LIN
Individual file: True
Individual rows: 1
Individual lap numbers: [np.int64(1)]
Telemetry files: 1
Telemetry filenames: ['1_tel.json']
Session lap file: EXISTS
Session total rows: 1214
Session records for driver: 0
Session lap numbers: []

------------------------------------------------------------
2026 | Chinese Grand Prix | Race | ALB
Individual file: True
Individual rows: 1
Individual lap numbers: [np.int64(1)]
Telemetry files: 1
Telemetry filenames: ['1_tel.jso

,event,session,driver,max_individual_lap,telemetry_lap,samples
0,Australian Grand Prix,Race,BOT,17,18,0
1,Austrian Grand Prix,Race,SAI,25,26,0
2,Azerbaijan Grand Prix,Practice 1,HUL,23,24,0
3,Azerbaijan Grand Prix,Race,GAS,37,38,0
4,Azerbaijan Grand Prix,Race,NOR,37,38,0
5,Bahrain Grand Prix,Practice 2,BOR,26,27,0
6,Bahrain Grand Prix,Race,ALB,43,44,0
7,Bahrain Grand Prix,Race,BOT,9,10,0
8,Bahrain Grand Prix,Race,RUS,51,52,0
9,Barcelona Grand Prix,Race,ALO,39,40,0



Event distribution:
event
Bahrain Grand Prix       4
Monaco Grand Prix        4
Canadian Grand Prix      4
Azerbaijan Grand Prix    3
Barcelona Grand Prix     2
Singapore Grand Prix     2
Australian Grand Prix    1
Austrian Grand Prix      1
British Grand Prix       1
Japanese Grand Prix      1
Hungarian Grand Prix     1
Miami Grand Prix         1
Spanish Grand Prix       1
Name: count, dtype: int64

Driver distribution:
driver
BOT    3
STR    3
GAS    2
HUL    2
RUS    2
ALB    2
BEA    2
NOR    2
BOR    1
SAI    1
ALO    1
ANT    1
PIA    1
LAW    1
PER    1
COL    1
Name: count, dtype: int64

STEP 2.9A.4 COMPLETED
Eight affected driver records investigated.
Pre-season raw JSON comparison performed.
Telemetry-only final laps inspected.
Original JSON files remain unchanged.

NO AUTOMATIC DATA CLEANING PERFORMED.


In [17]:

# ============================================================
# FOR1SIGHT STEP 2.9A.5
# JSON SCHEMA AND RECORD INVESTIGATION
# ============================================================

import json
import pandas as pd
import re
from pathlib import Path
from collections import Counter

print("=" * 75)
print("FOR1SIGHT STEP 2.9A.5")
print("JSON SCHEMA AND RECORD INVESTIGATION")
print("=" * 75)

# ------------------------------------------------------------
# 1. PATH CONFIGURATION
# ------------------------------------------------------------

RAW_ROOT = Path(
    r"C:\Users\ayepd\Desktop\For1Sight Project\raw_hf_data"
)

REPORT_DIR = RAW_ROOT.parent / "audit_reports"

driver_df = pd.read_csv(
    REPORT_DIR / "step2_9_driver_coverage.csv"
)

def normalize(text):
    return re.sub(
        r"[^a-z0-9]",
        "",
        str(text).lower()
    )

def find_directory(parent, name):

    if not parent.exists():
        return None

    folders = [
        p for p in parent.iterdir()
        if p.is_dir()
    ]

    target = normalize(name)

    matches = [
        p for p in folders
        if normalize(p.name) == target
    ]

    return matches[0] if matches else None


def read_json(path):

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


# ------------------------------------------------------------
# 2. GENERIC JSON STRUCTURE INSPECTOR
# ------------------------------------------------------------

def inspect_json_structure(obj, path="$", depth=0):

    indent = "  " * depth

    if isinstance(obj, dict):

        print(
            f"{indent}{path}: dict "
            f"({len(obj)} keys)"
        )

        for key, value in list(obj.items())[:20]:

            inspect_json_structure(
                value,
                f"{path}.{key}",
                depth + 1
            )

    elif isinstance(obj, list):

        print(
            f"{indent}{path}: list "
            f"(length={len(obj)})"
        )

        if len(obj) > 0:

            first = obj[0]

            print(
                f"{indent}First item type:",
                type(first).__name__
            )

            if isinstance(first, (str, int, float, bool)):
                print(
                    f"{indent}Sample:",
                    obj[:5]
                )

            elif depth < 4:

                inspect_json_structure(
                    first,
                    f"{path}[0]",
                    depth + 1
                )

    else:

        print(
            f"{indent}{path}: "
            f"{type(obj).__name__} = {str(obj)[:150]}"
        )


# ------------------------------------------------------------
# 3. INSPECT THE EIGHT INDIVIDUAL LAP RECORDS
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("3. INDIVIDUAL LAP RECORD INSPECTION")
print("=" * 75)

targets = [
    (2026, "Australian Grand Prix", "Race", "HUL"),
    (2026, "Canadian Grand Prix", "Race", "LIN"),
    (2026, "Chinese Grand Prix", "Race", "ALB"),
    (2026, "Chinese Grand Prix", "Race", "BOR"),
    (2026, "Chinese Grand Prix", "Race", "NOR"),
    (2026, "Chinese Grand Prix", "Race", "PIA"),
    (2026, "Miami Grand Prix", "Sprint", "HUL"),
    (2026, "Miami Grand Prix", "Sprint", "LIN"),
]

for year, event, session, driver in targets:

    event_dir = find_directory(
        RAW_ROOT / str(year),
        event
    )

    if event_dir is None:
        print("Missing event:", event)
        continue

    session_dir = find_directory(
        event_dir,
        session
    )

    driver_dir = find_directory(
        session_dir,
        driver
    ) if session_dir else None

    if driver_dir is None:
        print("Missing driver directory:", driver)
        continue

    individual_path = driver_dir / "laptimes.json"

    if not individual_path.exists():
        continue

    print("\n" + "-" * 60)
    print(f"{event} | {session} | {driver}")

    individual = read_json(individual_path)

    inspect_json_structure(individual)

    # Inspect the session file
    session_path = (
        session_dir / "session_laptimes.json"
    )

    if session_path.exists():

        session_data = read_json(session_path)

        print("\nSESSION JSON KEYS:")
        print(list(session_data.keys()))

        # Inspect the driver identifier field
        for key in [
            "drv",
            "driver_code",
            "Driver",
            "driver"
        ]:

            if key in session_data:

                values = session_data[key]

                if isinstance(values, list):

                    print(
                        f"\nDriver field: {key}"
                    )

                    print(
                        "Unique values:",
                        Counter(
                            str(x) for x in values
                        ).most_common(30)
                    )


# ------------------------------------------------------------
# 4. INVESTIGATE TELEMETRY JSON STRUCTURE
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("4. TELEMETRY JSON STRUCTURE")
print("=" * 75)

extra = driver_df[
    driver_df["Extra_Telemetry"] > 0
].copy()

inspected = 0

for _, row in extra.iterrows():

    event_dir = find_directory(
        RAW_ROOT / "2026",
        row["Event"]
    )

    if event_dir is None:
        continue

    session_dir = find_directory(
        event_dir,
        row["Session"]
    )

    if session_dir is None:
        continue

    driver_dir = find_directory(
        session_dir,
        row["Driver"]
    )

    if driver_dir is None:
        continue

    lap_values = str(
        row["Extra_Telemetry_Lap_Numbers"]
    )

    numbers = re.findall(
        r"\d+",
        lap_values
    )

    if not numbers:
        continue

    lap_number = int(numbers[0])

    telemetry_path = (
        driver_dir / f"{lap_number}_tel.json"
    )

    if not telemetry_path.exists():
        continue

    data = read_json(telemetry_path)

    print("\n" + "-" * 60)
    print(
        f"{row['Event']} | "
        f"{row['Session']} | "
        f"{row['Driver']} | "
        f"Lap {lap_number}"
    )

    print(
        "File size:",
        round(telemetry_path.stat().st_size / 1024, 2),
        "KB"
    )

    print(
        "Root type:",
        type(data).__name__
    )

    inspect_json_structure(data)

    inspected += 1

    # Inspect three files first to understand schema
    if inspected >= 3:
        break


# ------------------------------------------------------------
# 5. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 2.9A.5 COMPLETED")
print("=" * 75)

print("Individual lap JSON records inspected:", len(targets))
print("Telemetry JSON samples inspected:", inspected)

print("No source data modified.")


FOR1SIGHT STEP 2.9A.5
JSON SCHEMA AND RECORD INVESTIGATION

3. INDIVIDUAL LAP RECORD INSPECTION

------------------------------------------------------------
Australian Grand Prix | Race | HUL
$: dict (42 keys)
  $.time: list (length=1)
  First item type: str
  Sample: ['None']
  $.lap: list (length=1)
  First item type: int
  Sample: [1]
  $.compound: list (length=1)
  First item type: str
  Sample: ['HARD']
  $.stint: list (length=1)
  First item type: int
  Sample: [1]
  $.s1: list (length=1)
  First item type: str
  Sample: ['None']
  $.s2: list (length=1)
  First item type: str
  Sample: ['None']
  $.s3: list (length=1)
  First item type: str
  Sample: ['None']
  $.ms1: list (length=1)
  First item type: str
  Sample: ['None']
  $.ms2: list (length=1)
  First item type: str
  Sample: ['None']
  $.ms3: list (length=1)
  First item type: str
  Sample: ['777777774']
  $.life: list (length=1)
  First item type: int
  Sample: [1]
  $.pos: list (length=1)
  First item type: str
  Sample

In [18]:

# ============================================================
# FOR1SIGHT STEP 2.9A.6
# FULL TELEMETRY QUALITY INVESTIGATION
# ============================================================

import json
import re
import pandas as pd
import numpy as np
from pathlib import Path

RAW_ROOT = Path(
    r"C:\Users\ayepd\Desktop\For1Sight Project\raw_hf_data"
)

REPORT_DIR = RAW_ROOT.parent / "audit_reports"
REPORT_DIR.mkdir(exist_ok=True)

driver_df = pd.read_csv(
    REPORT_DIR / "step2_9_driver_coverage.csv"
)


def normalize(text):
    return re.sub(
        r"[^a-z0-9]",
        "",
        str(text).lower()
    )


def find_directory(parent, name):

    if not parent.exists():
        return None

    matches = [
        p for p in parent.iterdir()
        if p.is_dir()
        and normalize(p.name) == normalize(name)
    ]

    return matches[0] if matches else None


def read_json(path):

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


# ============================================================
# 1. INVESTIGATE ALL 26 EXTRA TELEMETRY FILES
# ============================================================

print("=" * 75)
print("1. FULL TELEMETRY QUALITY INVESTIGATION")
print("=" * 75)

extra = driver_df[
    driver_df["Extra_Telemetry"] > 0
].copy()

telemetry_results = []

for _, row in extra.iterrows():

    event_dir = find_directory(
        RAW_ROOT / "2026",
        row["Event"]
    )

    if event_dir is None:
        continue

    session_dir = find_directory(
        event_dir,
        row["Session"]
    )

    if session_dir is None:
        continue

    driver_dir = find_directory(
        session_dir,
        row["Driver"]
    )

    if driver_dir is None:
        continue

    lap_numbers = re.findall(
        r"\d+",
        str(row["Extra_Telemetry_Lap_Numbers"])
    )

    for lap_number in lap_numbers:

        telemetry_path = (
            driver_dir /
            f"{lap_number}_tel.json"
        )

        if not telemetry_path.exists():
            continue

        data = read_json(telemetry_path)

        # IMPORTANT:
        # Actual telemetry is nested under "tel".
        tel = data.get("tel", {})

        if not isinstance(tel, dict):
            continue

        time = pd.to_numeric(
            pd.Series(tel.get("time", [])),
            errors="coerce"
        )

        speed = pd.to_numeric(
            pd.Series(tel.get("speed", [])),
            errors="coerce"
        )

        rpm = pd.to_numeric(
            pd.Series(tel.get("rpm", [])),
            errors="coerce"
        )

        distance = pd.to_numeric(
            pd.Series(tel.get("distance", [])),
            errors="coerce"
        )

        throttle = pd.to_numeric(
            pd.Series(tel.get("throttle", [])),
            errors="coerce"
        )

        brake = pd.to_numeric(
            pd.Series(tel.get("brake", [])),
            errors="coerce"
        )

        # Check all array lengths
        array_lengths = {
            key: len(value)
            for key, value in tel.items()
            if isinstance(value, list)
        }

        unique_lengths = set(
            array_lengths.values()
        )

        sample_count = len(time)

        # Check whether measurements contain useful values
        speed_nonzero = (
            (speed.abs() > 0.001).sum()
        )

        rpm_nonzero = (
            (rpm.abs() > 0.001).sum()
        )

        distance_nonzero = (
            (distance.abs() > 0.001).sum()
        )

        # Classify preliminary quality
        if (
            sample_count <= 5
            and speed_nonzero == 0
            and distance_nonzero == 0
        ):
            classification = "Near-empty / zero placeholder"

        elif (
            sample_count > 5
            and (
                speed_nonzero > 0
                or distance_nonzero > 0
            )
        ):
            classification = "Contains non-zero telemetry"

        else:
            classification = "Requires further inspection"

        result = {
            "Event": row["Event"],
            "Session": row["Session"],
            "Driver": row["Driver"],
            "Telemetry_Lap": int(lap_number),

            "Samples": sample_count,

            "Time_Start": time.min(),
            "Time_End": time.max(),

            "Speed_Min": speed.min(),
            "Speed_Max": speed.max(),
            "Speed_Average": speed.mean(),

            "RPM_Min": rpm.min(),
            "RPM_Max": rpm.max(),

            "Distance_Max": distance.max(),

            "Throttle_Average": throttle.mean(),
            "Brake_Average": brake.mean(),

            "Speed_Nonzero_Samples": int(speed_nonzero),
            "RPM_Nonzero_Samples": int(rpm_nonzero),

            "Distance_Nonzero_Samples": int(distance_nonzero),

            "Array_Lengths_Consistent": (
                len(unique_lengths) <= 1
            ),

            "Array_Lengths": str(array_lengths),

            "Classification": classification,

            "File_Size_KB": round(
                telemetry_path.stat().st_size / 1024,
                2
            )
        }

        telemetry_results.append(result)


telemetry_df = pd.DataFrame(
    telemetry_results
)

print("\nTOTAL FILES INVESTIGATED:")
print(len(telemetry_df))

print("\nCLASSIFICATION SUMMARY:")

display(
    telemetry_df["Classification"]
    .value_counts()
    .rename_axis("Classification")
    .reset_index(name="Count")
)

print("\nCOMPLETE TELEMETRY REPORT:")

display(
    telemetry_df[
        [
            "Event",
            "Session",
            "Driver",
            "Telemetry_Lap",
            "Samples",
            "Speed_Max",
            "RPM_Max",
            "Distance_Max",
            "Array_Lengths_Consistent",
            "Classification"
        ]
    ]
)

telemetry_df.to_csv(
    REPORT_DIR / "step2_9A6_telemetry_quality.csv",
    index=False
)


# ============================================================
# 2. INVESTIGATE THE EIGHT INCOMPLETE LAP RECORDS
# ============================================================

print("\n" + "=" * 75)
print("2. INCOMPLETE LAP RECORD INVESTIGATION")
print("=" * 75)

targets = [
    (2026, "Australian Grand Prix", "Race", "HUL"),
    (2026, "Canadian Grand Prix", "Race", "LIN"),
    (2026, "Chinese Grand Prix", "Race", "ALB"),
    (2026, "Chinese Grand Prix", "Race", "BOR"),
    (2026, "Chinese Grand Prix", "Race", "NOR"),
    (2026, "Chinese Grand Prix", "Race", "PIA"),
    (2026, "Miami Grand Prix", "Sprint", "HUL"),
    (2026, "Miami Grand Prix", "Sprint", "LIN"),
]

record_results = []

fields = [
    "lap",
    "time",
    "compound",
    "stint",
    "s1",
    "s2",
    "s3",
    "pos",
    "status",
    "pb",
    "sesT",
    "drv",
    "dNum",
    "pout",
    "pin"
]

for year, event, session, driver in targets:

    event_dir = find_directory(
        RAW_ROOT / str(year),
        event
    )

    if event_dir is None:
        continue

    session_dir = find_directory(
        event_dir,
        session
    )

    driver_dir = (
        find_directory(session_dir, driver)
        if session_dir else None
    )

    if driver_dir is None:
        continue

    individual_path = (
        driver_dir / "laptimes.json"
    )

    if not individual_path.exists():
        continue

    data = read_json(individual_path)

    record = {
        "Season": year,
        "Event": event,
        "Session": session,
        "Driver": driver
    }

    for field in fields:

        value = data.get(field, [])

        record[field] = (
            value[0]
            if isinstance(value, list) and value
            else None
        )

    record_results.append(record)


records_df = pd.DataFrame(record_results)

print("\nEIGHT INDIVIDUAL RECORDS:")

display(records_df)

records_df.to_csv(
    REPORT_DIR / "step2_9A6_incomplete_laps.csv",
    index=False
)

print("\nSTATUS DISTRIBUTION:")

display(
    records_df.groupby("status")
    .size()
    .reset_index(name="Count")
)


# ============================================================
# 3. INVESTIGATE STATUS DEFINITION IN DATA DICTIONARY
# ============================================================

print("\n" + "=" * 75)
print("3. STATUS FIELD DEFINITION")
print("=" * 75)

dictionary_path = (
    RAW_ROOT / "2026" / "data_dictionary.json"
)

def find_status_definition(obj, path="$"):

    if isinstance(obj, dict):

        for key, value in obj.items():

            current_path = f"{path}.{key}"

            if normalize(key) == "status":

                print("\nFOUND STATUS DEFINITION:")
                print("Location:", current_path)
                print(json.dumps(
                    value,
                    indent=2,
                    ensure_ascii=False
                )[:3000])

            find_status_definition(
                value,
                current_path
            )

    elif isinstance(obj, list):

        for index, value in enumerate(obj[:20]):

            find_status_definition(
                value,
                f"{path}[{index}]"
            )


if dictionary_path.exists():

    dictionary = read_json(dictionary_path)

    find_status_definition(dictionary)

else:

    print("Data dictionary not found.")


# ============================================================
# 4. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 75)
print("STEP 2.9A.6 COMPLETED")
print("=" * 75)

print(
    "Telemetry files investigated:",
    len(telemetry_df)
)

print(
    "Individual incomplete records:",
    len(records_df)
)

print(
    "Reports saved to:",
    REPORT_DIR
)

print("\nOriginal raw JSON files remain unchanged.")


1. FULL TELEMETRY QUALITY INVESTIGATION

TOTAL FILES INVESTIGATED:
26

CLASSIFICATION SUMMARY:


,Classification,Count
0,Near-empty / zero placeholder,17
1,Contains non-zero telemetry,7
2,Requires further inspection,2



COMPLETE TELEMETRY REPORT:


,Event,Session,Driver,Telemetry_Lap,Samples,Speed_Max,RPM_Max,Distance_Max,Array_Lengths_Consistent,Classification
0,Australian Grand Prix,Race,BOT,18,3,0.0,0.0,0.000000,False,Near-empty / zero placeholder
1,Austrian Grand Prix,Race,SAI,26,1155,188.0,11574.0,7833.333330,True,Contains non-zero telemetry
2,Azerbaijan Grand Prix,Practice 1,HUL,24,1132,282.0,11176.0,11749.999990,True,Contains non-zero telemetry
3,Azerbaijan Grand Prix,Race,GAS,38,4,0.0,0.0,0.000000,False,Near-empty / zero placeholder
4,Azerbaijan Grand Prix,Race,NOR,38,3,0.0,0.0,0.000000,False,Near-empty / zero placeholder
5,Bahrain Grand Prix,Practice 2,BOR,27,4,0.0,0.0,0.000000,False,Near-empty / zero placeholder
6,Bahrain Grand Prix,Race,ALB,44,3,0.0,0.0,0.000000,False,Near-empty / zero placeholder
7,Bahrain Grand Prix,Race,BOT,10,3,0.0,0.0,0.000000,False,Near-empty / zero placeholder
8,Bahrain Grand Prix,Race,RUS,52,3,0.0,0.0,0.000000,False,Near-empty / zero placeholder
9,Barcelona Grand Prix,Race,ALO,40,3,0.0,0.0,0.000000,False,Near-empty / zero placeholder



2. INCOMPLETE LAP RECORD INVESTIGATION

EIGHT INDIVIDUAL RECORDS:


,Season,Event,Session,Driver,lap,time,compound,stint,s1,s2,s3,pos,status,pb,sesT,drv,dNum,pout,pin
0,2026,Australian Grand Prix,Race,HUL,1,None,HARD,1,None,None,None,None,1,False,3831.908,HUL,27,None,None
1,2026,Canadian Grand Prix,Race,LIN,1,None,MEDIUM,1,None,None,None,None,1,False,3838.725,LIN,41,None,None
2,2026,Chinese Grand Prix,Race,ALB,1,None,HARD,1,None,None,None,None,12,False,3483.760,ALB,23,None,None
3,2026,Chinese Grand Prix,Race,BOR,1,None,HARD,1,None,None,None,None,12,False,3483.760,BOR,5,None,None
4,2026,Chinese Grand Prix,Race,NOR,1,None,HARD,1,None,None,None,None,12,False,3483.760,NOR,1,None,None
5,2026,Chinese Grand Prix,Race,PIA,1,None,MEDIUM,1,None,None,None,None,12,False,3483.760,PIA,81,None,None
6,2026,Miami Grand Prix,Sprint,HUL,1,None,MEDIUM,1,None,None,None,None,1,False,2745.655,HUL,27,None,None
7,2026,Miami Grand Prix,Sprint,LIN,1,None,MEDIUM,1,None,None,None,None,1,False,2745.655,LIN,41,None,None



STATUS DISTRIBUTION:


,status,Count
0,1,4
1,12,4



3. STATUS FIELD DEFINITION

FOUND STATUS DEFINITION:
Location: $.files.rcm.json.fields.status
{
  "full_name": "Status",
  "description": "Additional context about the message state",
  "source": "FastF1",
  "values": "Examples: \"DISABLED\", \"ENABLED\", depends on category"
}

FOUND STATUS DEFINITION:
Location: $.files.laptimes.json.fields.status
{
  "full_name": "Track Status",
  "description": "A string that contains track status numbers for all track status that occurred during this lap.",
  "source": "FastF1",
  "type": "String: \"1\", \"216\""
}

STEP 2.9A.6 COMPLETED
Telemetry files investigated: 26
Individual incomplete records: 8
Reports saved to: C:\Users\ayepd\Desktop\For1Sight Project\audit_reports

Original raw JSON files remain unchanged.


In [19]:

# ============================================================
# FOR1SIGHT STEP 2.9A.7
# TELEMETRY VALIDATION & RACE CONTROL RECONCILIATION
# ============================================================

import json
import pandas as pd
import numpy as np
from pathlib import Path

RAW_ROOT = Path(
    r"C:\Users\ayepd\Desktop\For1Sight Project\raw_hf_data"
)

REPORT_DIR = RAW_ROOT.parent / "audit_reports"

telemetry_file = (
    REPORT_DIR / "step2_9A6_telemetry_quality.csv"
)

records_file = (
    REPORT_DIR / "step2_9A6_incomplete_laps.csv"
)

telemetry_df = pd.read_csv(telemetry_file)
records_df = pd.read_csv(records_file)


def normalize(text):
    import re
    return re.sub(
        r"[^a-z0-9]",
        "",
        str(text).lower()
    )


def find_directory(parent, name):

    if not parent.exists():
        return None

    for folder in parent.iterdir():

        if (
            folder.is_dir()
            and normalize(folder.name) == normalize(name)
        ):
            return folder

    return None


def read_json(path):

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def numeric_values(values):

    return pd.to_numeric(
        pd.Series(values),
        errors="coerce"
    ).dropna()


# ============================================================
# PART A: TELEMETRY VALIDATION
# ============================================================

print("=" * 75)
print("PART A: TELEMETRY VALIDATION")
print("=" * 75)

channels = [
    "time",
    "speed",
    "rpm",
    "gear",
    "throttle",
    "brake",
    "drs",
    "distance",
    "rel_distance"
]

quality_results = []

for _, row in telemetry_df.iterrows():

    event_dir = find_directory(
        RAW_ROOT / "2026",
        row["Event"]
    )

    if event_dir is None:
        continue

    session_dir = find_directory(
        event_dir,
        row["Session"]
    )

    driver_dir = (
        find_directory(session_dir, row["Driver"])
        if session_dir else None
    )

    if driver_dir is None:
        continue

    lap = int(row["Telemetry_Lap"])

    path = driver_dir / f"{lap}_tel.json"

    if not path.exists():
        continue

    raw = read_json(path)
    tel = raw.get("tel", {})

    if not isinstance(tel, dict):
        continue

    arrays = {
        key: value
        for key, value in tel.items()
        if isinstance(value, list)
    }

    all_lengths = {
        key: len(value)
        for key, value in arrays.items()
    }

    core_lengths = {
        key: len(tel[key])
        for key in channels
        if isinstance(tel.get(key), list)
    }

    core_complete = (
        len(core_lengths) == len(channels)
    )

    core_consistent = (
        core_complete
        and len(set(core_lengths.values())) == 1
    )

    all_consistent = (
        len(set(all_lengths.values())) <= 1
    )

    numeric = {
        key: numeric_values(tel.get(key, []))
        for key in channels
    }

    time = numeric["time"]
    distance = numeric["distance"]
    speed = numeric["speed"]
    rpm = numeric["rpm"]
    throttle = numeric["throttle"]
    brake = numeric["brake"]

    time_monotonic = (
        time.diff().dropna().ge(0).all()
        if len(time) > 1 else None
    )

    distance_monotonic = (
        distance.diff().dropna().ge(0).all()
        if len(distance) > 1 else None
    )

    result = {
        "Event": row["Event"],
        "Session": row["Session"],
        "Driver": row["Driver"],
        "Lap": lap,

        "Time_Samples": len(time),
        "Core_Lengths_Consistent": core_consistent,
        "All_Array_Lengths_Consistent": all_consistent,

        "Time_Span": (
            time.max() - time.min()
            if len(time) else np.nan
        ),

        "Time_Monotonic": time_monotonic,
        "Distance_Monotonic": distance_monotonic,

        "Speed_Nonzero": int((speed.abs() > 0.001).sum()),
        "Speed_Average": speed.mean(),
        "Speed_Max": speed.max(),

        "RPM_Nonzero": int((rpm.abs() > 0.001).sum()),
        "RPM_Average": rpm.mean(),
        "RPM_Max": rpm.max(),

        "Throttle_Average": throttle.mean(),
        "Brake_Average": brake.mean(),

        "Distance_Min": distance.min(),
        "Distance_Max": distance.max(),

        "Distance_Negative_Count": int((distance < 0).sum()),

        "Core_Array_Lengths": str(core_lengths),
        "All_Array_Lengths": str(all_lengths)
    }

    quality_results.append(result)


quality_df = pd.DataFrame(quality_results)

print("\nTELEMETRY QUALITY SUMMARY")

display(
    quality_df[
        [
            "Event",
            "Driver",
            "Session",
            "Lap",
            "Time_Samples",
            "Core_Lengths_Consistent",
            "All_Array_Lengths_Consistent",
            "Time_Monotonic",
            "Distance_Monotonic",
            "Speed_Max",
            "RPM_Max",
            "Distance_Max"
        ]
    ]
)

quality_df.to_csv(
    REPORT_DIR / "step2_9A7_telemetry_validation.csv",
    index=False
)

print("\nCORE ARRAY CONSISTENCY:")
print(
    quality_df["Core_Lengths_Consistent"]
    .value_counts(dropna=False)
)

print("\nTIME MONOTONICITY:")
print(
    quality_df["Time_Monotonic"]
    .value_counts(dropna=False)
)

print("\nDISTANCE MONOTONICITY:")
print(
    quality_df["Distance_Monotonic"]
    .value_counts(dropna=False)
)


# ============================================================
# PART B: DATA DICTIONARY INVESTIGATION
# ============================================================

print("\n" + "=" * 75)
print("PART B: DATA DICTIONARY INVESTIGATION")
print("=" * 75)

dictionary_path = (
    RAW_ROOT / "2026" / "data_dictionary.json"
)

def find_fields(obj, targets, path="$"):

    if isinstance(obj, dict):

        for key, value in obj.items():

            current = f"{path}.{key}"

            if normalize(key) in targets:

                print("\nFIELD:", current)

                print(
                    json.dumps(
                        value,
                        indent=2,
                        ensure_ascii=False
                    )[:2500]
                )

            find_fields(
                value,
                targets,
                current
            )

    elif isinstance(obj, list):

        for i, value in enumerate(obj[:20]):

            find_fields(
                value,
                targets,
                f"{path}[{i}]"
            )


if dictionary_path.exists():

    dictionary = read_json(dictionary_path)

    find_fields(
        dictionary,
        {
            "sest",
            "status",
            "time",
            "lap",
            "pos"
        }
    )


# ============================================================
# PART C: RACE CONTROL JSON STRUCTURE
# ============================================================

print("\n" + "=" * 75)
print("PART C: RACE CONTROL JSON INVESTIGATION")
print("=" * 75)


def inspect_rcm_structure(obj, depth=0):

    indent = "  " * depth

    if isinstance(obj, dict):

        print(
            indent,
            "DICT:",
            list(obj.keys())[:25]
        )

        for key, value in list(obj.items())[:10]:

            if isinstance(value, (dict, list)):

                print(
                    indent,
                    "FIELD:",
                    key
                )

                if depth < 2:

                    inspect_rcm_structure(
                        value,
                        depth + 1
                    )

    elif isinstance(obj, list):

        print(
            indent,
            "LIST LENGTH:",
            len(obj)
        )

        if obj:

            print(
                indent,
                "FIRST RECORD:"
            )

            inspect_rcm_structure(
                obj[0],
                depth + 1
            )


# Only four unique event/session combinations
rcm_targets = (
    records_df[
        ["Event", "Session"]
    ]
    .drop_duplicates()
)

for _, row in rcm_targets.iterrows():

    event_dir = find_directory(
        RAW_ROOT / "2026",
        row["Event"]
    )

    if event_dir is None:
        continue

    session_dir = find_directory(
        event_dir,
        row["Session"]
    )

    if session_dir is None:
        continue

    rcm_path = session_dir / "rcm.json"

    print("\n" + "-" * 65)
    print(
        row["Event"],
        "|",
        row["Session"]
    )

    if not rcm_path.exists():

        print("RCM FILE NOT FOUND")
        continue

    rcm = read_json(rcm_path)

    print("File size (KB):", round(
        rcm_path.stat().st_size / 1024,
        2
    ))

    inspect_rcm_structure(rcm)


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 75)
print("STEP 2.9A.7 COMPLETED")
print("=" * 75)

print("Telemetry files validated:", len(quality_df))
print("RCM event/session combinations:", len(rcm_targets))

print("\nReports saved:")
print(
    REPORT_DIR / "step2_9A7_telemetry_validation.csv"
)

print("\nNo original JSON files modified.")


PART A: TELEMETRY VALIDATION

TELEMETRY QUALITY SUMMARY


,Event,Driver,Session,Lap,Time_Samples,Core_Lengths_Consistent,All_Array_Lengths_Consistent,Time_Monotonic,Distance_Monotonic,Speed_Max,RPM_Max,Distance_Max
0,Australian Grand Prix,BOT,Race,18,3,True,False,True,True,0.0,0.0,0.000000
1,Austrian Grand Prix,SAI,Race,26,1155,True,True,True,True,188.0,11574.0,7833.333330
2,Azerbaijan Grand Prix,HUL,Practice 1,24,1132,True,True,True,True,282.0,11176.0,11749.999990
3,Azerbaijan Grand Prix,GAS,Race,38,4,True,False,True,True,0.0,0.0,0.000000
4,Azerbaijan Grand Prix,NOR,Race,38,3,True,False,True,True,0.0,0.0,0.000000
5,Bahrain Grand Prix,BOR,Practice 2,27,4,True,False,True,True,0.0,0.0,0.000000
6,Bahrain Grand Prix,ALB,Race,44,3,True,False,True,True,0.0,0.0,0.000000
7,Bahrain Grand Prix,BOT,Race,10,3,True,False,True,True,0.0,0.0,0.000000
8,Bahrain Grand Prix,RUS,Race,52,3,True,False,True,True,0.0,0.0,0.000000
9,Barcelona Grand Prix,ALO,Race,40,3,True,False,True,True,0.0,0.0,0.000000



CORE ARRAY CONSISTENCY:
Core_Lengths_Consistent
True    26
Name: count, dtype: int64

TIME MONOTONICITY:
Time_Monotonic
True    26
Name: count, dtype: int64

DISTANCE MONOTONICITY:
Distance_Monotonic
True     25
False     1
Name: count, dtype: int64

PART B: DATA DICTIONARY INVESTIGATION

FIELD: $.files.rcm.json.fields.time
{
  "full_name": "Time",
  "description": "Session time when this message was issued",
  "source": "FastF1",
  "values": "Time"
}

FIELD: $.files.rcm.json.fields.status
{
  "full_name": "Status",
  "description": "Additional context about the message state",
  "source": "FastF1",
  "values": "Examples: \"DISABLED\", \"ENABLED\", depends on category"
}

FIELD: $.files.rcm.json.fields.lap
{
  "full_name": "Lap",
  "description": "Which lap number this message refers to",
  "source": "FastF1",
  "values": "Integer: lap number"
}

FIELD: $.files.tel.json.fields.time
{
  "full_name": "Time",
  "description": "Time when this data sample was recorded from start of the lap

In [20]:

# ============================================================
# FOR1SIGHT STEP 2.9A.8
# TELEMETRY COVERAGE & RACE CONTROL EVENT ANALYSIS
# ============================================================

import json
import pandas as pd
import numpy as np
import re
from pathlib import Path

RAW_ROOT = Path(
    r"C:\Users\ayepd\Desktop\For1Sight Project\raw_hf_data"
)

REPORT_DIR = RAW_ROOT.parent / "audit_reports"

REPORT_DIR.mkdir(exist_ok=True)

QUALITY_FILE = (
    REPORT_DIR / "step2_9A7_telemetry_validation.csv"
)

LAP_FILE = (
    REPORT_DIR / "step2_9A6_incomplete_laps.csv"
)

quality_df = pd.read_csv(QUALITY_FILE)
records_df = pd.read_csv(LAP_FILE)


def normalize(text):
    return re.sub(
        r"[^a-z0-9]",
        "",
        str(text).lower()
    )


def find_directory(parent, name):

    if not parent.exists():
        return None

    for folder in parent.iterdir():

        if (
            folder.is_dir()
            and normalize(folder.name) == normalize(name)
        ):
            return folder

    return None


def read_json(path):

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def numeric_array(values):

    return pd.to_numeric(
        pd.Series(values),
        errors="coerce"
    ).dropna()


def to_seconds(value):

    if pd.isna(value):
        return np.nan

    try:
        return float(value)

    except (ValueError, TypeError):
        pass

    try:
        return pd.to_timedelta(value).total_seconds()

    except Exception:
        return np.nan


# ============================================================
# PART A: TELEMETRY COVERAGE ANALYSIS
# ============================================================

print("=" * 75)
print("PART A: TELEMETRY COVERAGE ANALYSIS")
print("=" * 75)

coverage_results = []

# Inspect all 26 files, including unusual files.

for _, row in quality_df.iterrows():

    event_dir = find_directory(
        RAW_ROOT / "2026",
        row["Event"]
    )

    if event_dir is None:
        continue

    session_dir = find_directory(
        event_dir,
        row["Session"]
    )

    driver_dir = (
        find_directory(session_dir, row["Driver"])
        if session_dir else None
    )

    if driver_dir is None:
        continue

    lap = int(row["Lap"])

    path = driver_dir / f"{lap}_tel.json"

    if not path.exists():
        continue

    raw = read_json(path)
    tel = raw.get("tel", {})

    if not isinstance(tel, dict):
        continue

    time = numeric_array(tel.get("time", []))
    distance = numeric_array(tel.get("distance", []))
    relative = numeric_array(tel.get("rel_distance", []))
    speed = numeric_array(tel.get("speed", []))
    rpm = numeric_array(tel.get("rpm", []))
    throttle = numeric_array(tel.get("throttle", []))
    brake = numeric_array(tel.get("brake", []))

    if len(time) > 0:

        time_start = time.min()
        time_end = time.max()
        time_duration = time_end - time_start

    else:

        time_start = np.nan
        time_end = np.nan
        time_duration = np.nan

    if len(relative) > 0:

        relative_start = relative.min()
        relative_end = relative.max()

        relative_coverage = (
            relative_end - relative_start
        )

    else:

        relative_start = np.nan
        relative_end = np.nan
        relative_coverage = np.nan

    if len(distance) > 1:

        distance_increasing = (
            distance.diff().dropna() >= -0.000001
        ).all()

    else:

        distance_increasing = None

    result = {

        "Event": row["Event"],
        "Session": row["Session"],
        "Driver": row["Driver"],
        "Lap": lap,

        "Samples": len(time),

        "Time_Start": time_start,
        "Time_End": time_end,
        "Time_Duration": time_duration,

        "Relative_Start": relative_start,
        "Relative_End": relative_end,
        "Relative_Coverage": relative_coverage,

        "Distance_Start": distance.min() if len(distance) else np.nan,
        "Distance_End": distance.max() if len(distance) else np.nan,

        "Distance_Increasing": distance_increasing,

        "Speed_Mean": speed.mean(),
        "Speed_Median": speed.median(),
        "Speed_P90": speed.quantile(0.90) if len(speed) else np.nan,
        "Speed_Max": speed.max(),

        "RPM_Mean": rpm.mean(),
        "RPM_Max": rpm.max(),

        "Throttle_Mean": throttle.mean(),
        "Brake_Mean": brake.mean(),

        "Speed_Above_100_Count": int(
            (speed > 100).sum()
        ),

        "Nonzero_Speed_Count": int(
            (speed.abs() > 0.001).sum()
        )
    }

    coverage_results.append(result)


coverage_df = pd.DataFrame(coverage_results)

print("\nTELEMETRY COVERAGE RESULTS:")

display(
    coverage_df[
        [
            "Event",
            "Session",
            "Driver",
            "Lap",
            "Samples",
            "Time_Duration",
            "Relative_Start",
            "Relative_End",
            "Relative_Coverage",
            "Distance_End",
            "Speed_Mean",
            "Speed_Max"
        ]
    ]
)

coverage_df.to_csv(
    REPORT_DIR / "step2_9A8_telemetry_coverage.csv",
    index=False
)

print("\nTELEMETRY SAMPLE SUMMARY:")

display(
    coverage_df[
        [
            "Samples",
            "Time_Duration",
            "Relative_Coverage",
            "Speed_Mean",
            "Speed_Max"
        ]
    ].describe()
)


# ============================================================
# PART B: RACE CONTROL MESSAGE EXTRACTION
# ============================================================

print("\n" + "=" * 75)
print("PART B: RACE CONTROL MESSAGE EXTRACTION")
print("=" * 75)

rcm_targets = (
    records_df[
        ["Season", "Event", "Session"]
    ]
    .drop_duplicates()
)

rcm_results = []

for _, row in rcm_targets.iterrows():

    event_dir = find_directory(
        RAW_ROOT / str(int(row["Season"])),
        row["Event"]
    )

    if event_dir is None:
        continue

    session_dir = find_directory(
        event_dir,
        row["Session"]
    )

    if session_dir is None:
        continue

    path = session_dir / "rcm.json"

    if not path.exists():
        print("Missing:", path)
        continue

    raw = read_json(path)

    # RCM uses column-oriented arrays.
    # Series handles fields with different lengths.

    rcm_df = pd.DataFrame({
        key: pd.Series(value)
        for key, value in raw.items()
        if isinstance(value, list)
    })

    rcm_df.insert(0, "Season", row["Season"])
    rcm_df.insert(1, "Event", row["Event"])
    rcm_df.insert(2, "Session", row["Session"])

    rcm_df["time_seconds"] = (
        rcm_df["time"].apply(to_seconds)
        if "time" in rcm_df.columns
        else np.nan
    )

    rcm_results.append(rcm_df)

    print("\n" + "-" * 65)

    print(
        row["Event"],
        "|",
        row["Session"]
    )

    print("Total RCM messages:", len(rcm_df))

    print("\nMESSAGE CATEGORY DISTRIBUTION:")

    if "cat" in rcm_df.columns:

        display(
            rcm_df["cat"]
            .value_counts(dropna=False)
            .reset_index()
        )

    print("\nTRACK STATUS / FLAG DISTRIBUTION:")

    for field in ["status", "flag", "scope"]:

        if field in rcm_df.columns:

            print("\n", field)

            print(
                rcm_df[field]
                .value_counts(dropna=False)
                .head(15)
            )

    print("\nFIRST 10 MESSAGES:")

    display(
        rcm_df[
            [
                c for c in [
                    "time",
                    "cat",
                    "msg",
                    "status",
                    "flag",
                    "scope",
                    "sector",
                    "dNum",
                    "lap"
                ]
                if c in rcm_df.columns
            ]
        ].head(10)
    )


if rcm_results:

    all_rcm = pd.concat(
        rcm_results,
        ignore_index=True
    )

else:

    all_rcm = pd.DataFrame()


# ============================================================
# PART C: MATCH RACE CONTROL TO INCOMPLETE LAP RECORDS
# ============================================================

print("\n" + "=" * 75)
print("PART C: RACE CONTROL / LAP TIME MATCHING")
print("=" * 75)

context_results = []

if not all_rcm.empty:

    for _, lap_row in records_df.iterrows():

        ses_time = to_seconds(
            lap_row["sesT"]
        )

        if pd.isna(ses_time):
            continue

        matching = all_rcm[
            (all_rcm["Event"] == lap_row["Event"])
            &
            (all_rcm["Session"] == lap_row["Session"])
        ].copy()

        matching = matching.dropna(
            subset=["time_seconds"]
        )

        if matching.empty:
            continue

        matching["Time_Difference"] = (
            matching["time_seconds"] - ses_time
        )

        matching["Absolute_Difference"] = (
            matching["Time_Difference"].abs()
        )

        # Select messages within 90 seconds
        nearby = matching[
            matching["Absolute_Difference"] <= 90
        ].copy()

        nearby["Target_Driver"] = lap_row["Driver"]
        nearby["Target_Lap"] = lap_row["lap"]
        nearby["Target_Session_Time"] = ses_time

        if not nearby.empty:

            context_results.append(nearby)

            print("\n" + "-" * 65)

            print(
                lap_row["Event"],
                "|",
                lap_row["Driver"],
                "|",
                "Lap",
                lap_row["lap"]
            )

            print(
                "Lap session time:",
                ses_time
            )

            print(
                "Nearby RCM messages:",
                len(nearby)
            )

            display(
                nearby.sort_values(
                    "Absolute_Difference"
                )[
                    [
                        c for c in [
                            "time",
                            "Time_Difference",
                            "cat",
                            "msg",
                            "status",
                            "flag",
                            "scope",
                            "sector",
                            "dNum",
                            "lap"
                        ]
                        if c in nearby.columns
                    ]
                ].head(15)
            )

if context_results:

    context_df = pd.concat(
        context_results,
        ignore_index=True
    )

    context_df.to_csv(
        REPORT_DIR / "step2_9A8_rcm_lap_context.csv",
        index=False
    )

else:

    print(
        "No matching messages within the selected "
        "90-second window, or RCM timestamps "
        "could not be interpreted."
    )


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 75)
print("STEP 2.9A.8 COMPLETED")
print("=" * 75)

print(
    "Telemetry files analysed:",
    len(coverage_df)
)

print(
    "RCM event/session datasets:",
    len(all_rcm)
)

print(
    "Original JSON files were not modified."
)

print(
    "Reports saved in:",
    REPORT_DIR
)


PART A: TELEMETRY COVERAGE ANALYSIS

TELEMETRY COVERAGE RESULTS:


,Event,Session,Driver,Lap,Samples,Time_Duration,Relative_Start,Relative_End,Relative_Coverage,Distance_End,Speed_Mean,Speed_Max
0,Australian Grand Prix,Race,BOT,18,3,0.280,NaN,NaN,NaN,0.000000,0.0,0.0
1,Austrian Grand Prix,Race,SAI,26,1155,150.000,8.522870e-10,0.998781,0.998781,7833.333330,188.0,188.0
2,Azerbaijan Grand Prix,Practice 1,HUL,24,1132,150.000,4.110041e-10,0.999800,0.999800,11749.999990,282.0,282.0
3,Azerbaijan Grand Prix,Race,GAS,38,4,0.400,NaN,NaN,NaN,0.000000,0.0,0.0
4,Azerbaijan Grand Prix,Race,NOR,38,3,0.401,NaN,NaN,NaN,0.000000,0.0,0.0
5,Bahrain Grand Prix,Practice 2,BOR,27,4,0.160,NaN,NaN,NaN,0.000000,0.0,0.0
6,Bahrain Grand Prix,Race,ALB,44,3,0.160,NaN,NaN,NaN,0.000000,0.0,0.0
7,Bahrain Grand Prix,Race,BOT,10,3,0.280,NaN,NaN,NaN,0.000000,0.0,0.0
8,Bahrain Grand Prix,Race,RUS,52,3,0.280,NaN,NaN,NaN,0.000000,0.0,0.0
9,Barcelona Grand Prix,Race,ALO,40,3,0.321,NaN,NaN,NaN,0.000000,0.0,0.0



TELEMETRY SAMPLE SUMMARY:


,Samples,Time_Duration,Relative_Coverage,Speed_Mean,Speed_Max
count,26.000000,26.000000,9.000000,26.000000,26.000000
mean,306.384615,40.578500,0.776992,43.923077,43.923077
std,509.521821,67.731667,0.440514,90.725486,90.725486
min,3.000000,0.159000,0.000000,0.000000,0.000000
25%,3.000000,0.210000,0.997958,0.000000,0.000000
50%,3.000000,0.280000,0.998781,0.000000,0.000000
75%,834.250000,112.600250,0.998968,7.500000,7.500000
max,1155.000000,150.000000,0.999800,282.000000,282.000000



PART B: RACE CONTROL MESSAGE EXTRACTION

-----------------------------------------------------------------
Australian Grand Prix | Race
Total RCM messages: 167

MESSAGE CATEGORY DISTRIBUTION:


,cat,count
0,Flag,124
1,Other,37
2,SafetyCar,6


... (11 outputs left out)

,cat,count
0,Flag,53
1,Other,29



TRACK STATUS / FLAG DISTRIBUTION:

 status
status
None    82
Name: count, dtype: int64

 flag
flag
None               29
CLEAR              21
DOUBLE YELLOW      19
YELLOW              9
GREEN               2
BLACK AND WHITE     1
CHEQUERED           1
Name: count, dtype: int64

 scope
scope
Sector    49
None      29
Track      3
Driver     1
Name: count, dtype: int64

FIRST 10 MESSAGES:


,time,cat,msg,status,flag,scope,sector,dNum,lap
0,2026-05-02T15:30:01.000000000,Flag,GREEN LIGHT - PIT EXIT OPEN,None,GREEN,Track,None,None,1
1,2026-05-02T15:35:01.000000000,Other,PIT EXIT CLOSED,None,None,None,None,None,1
2,2026-05-02T15:36:43.000000000,Flag,DOUBLE YELLOW IN TRACK SECTOR 19,None,DOUBLE YELLOW,Sector,19.0,None,1
3,2026-05-02T15:36:47.000000000,Flag,YELLOW IN TRACK SECTOR 18,None,YELLOW,Sector,18.0,None,1
4,2026-05-02T15:38:32.000000000,Other,FIA STEWARDS: INCIDENT INVOLVING CAR 63 (RUS) ...,None,None,None,None,None,1
5,2026-05-02T15:38:43.000000000,Other,FIA STEWARDS: INCIDENT INVOLVING CAR 63 (RUS) ...,None,None,None,None,None,1
6,2026-05-02T15:39:58.000000000,Flag,DOUBLE YELLOW IN TRACK SECTOR 15,None,DOUBLE YELLOW,Sector,15.0,None,1
7,2026-05-02T15:39:58.000000000,Flag,DOUBLE YELLOW IN TRACK SECTOR 16,None,DOUBLE YELLOW,Sector,16.0,None,1
8,2026-05-02T15:39:58.000000000,Flag,DOUBLE YELLOW IN TRACK SECTOR 17,None,DOUBLE YELLOW,Sector,17.0,None,1
9,2026-05-02T15:40:01.000000000,Flag,CLEAR IN TRACK SECTOR 18,None,CLEAR,Sector,18.0,None,1



PART C: RACE CONTROL / LAP TIME MATCHING
No matching messages within the selected 90-second window, or RCM timestamps could not be interpreted.

STEP 2.9A.8 COMPLETED
Telemetry files analysed: 26
RCM event/session datasets: 621
Original JSON files were not modified.
Reports saved in: C:\Users\ayepd\Desktop\For1Sight Project\audit_reports


In [21]:

# ============================================================
# FOR1SIGHT STEP 2.9A.9
# TELEMETRY STRUCTURE & TIMESTAMP INVESTIGATION
# ============================================================

import json
import pandas as pd
import numpy as np
from pathlib import Path

RAW_ROOT = Path(
    r"C:\Users\ayepd\Desktop\For1Sight Project\raw_hf_data"
)

REPORT_DIR = RAW_ROOT.parent / "audit_reports"

QUALITY_FILE = (
    REPORT_DIR / "step2_9A7_telemetry_validation.csv"
)

quality_df = pd.read_csv(QUALITY_FILE)


def normalize(text):
    import re
    return re.sub(r"[^a-z0-9]", "", str(text).lower())


def find_directory(parent, name):

    if not parent.exists():
        return None

    for folder in parent.iterdir():

        if (
            folder.is_dir()
            and normalize(folder.name) == normalize(name)
        ):
            return folder

    return None


def read_json(path):

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


# ============================================================
# PART A: INVESTIGATE ACTUAL TELEMETRY ARRAYS
# ============================================================

print("=" * 75)
print("PART A: TELEMETRY JSON STRUCTURE")
print("=" * 75)

# Select one complete file and one incomplete file.

full_rows = quality_df[
    quality_df["Time_Samples"] >= 1000
]

short_rows = quality_df[
    quality_df["Time_Samples"] <= 4
]

selected_rows = []

if not full_rows.empty:
    selected_rows.append(
        ("COMPLETE TELEMETRY", full_rows.iloc[0])
    )

if not short_rows.empty:
    selected_rows.append(
        ("INCOMPLETE TELEMETRY", short_rows.iloc[0])
    )


for label, row in selected_rows:

    event_dir = find_directory(
        RAW_ROOT / "2026",
        row["Event"]
    )

    session_dir = find_directory(
        event_dir,
        row["Session"]
    )

    driver_dir = find_directory(
        session_dir,
        row["Driver"]
    )

    lap = int(row["Lap"])

    path = driver_dir / f"{lap}_tel.json"

    print("\n" + "=" * 65)
    print(label)
    print("=" * 65)

    print("File:", path)

    raw = read_json(path)

    print("\nROOT JSON KEYS:")
    print(list(raw.keys()))

    tel = raw.get("tel", raw)

    print("\nTELEMETRY OBJECT TYPE:")
    print(type(tel).__name__)

    if not isinstance(tel, dict):
        print("Unexpected telemetry structure.")
        continue

    print("\nTELEMETRY FIELD LENGTHS:")

    for key, value in tel.items():

        if isinstance(value, list):
            print(f"{key:20} | Length: {len(value)}")

        else:
            print(
                f"{key:20} | Type: {type(value).__name__}"
            )

    print("\nIMPORTANT NUMERICAL FIELD INVESTIGATION:")

    for field in [
        "time",
        "speed",
        "rpm",
        "distance",
        "rel_distance",
        "throttle",
        "brake",
        "gear"
    ]:

        value = tel.get(field)

        print("\nFIELD:", field)

        if isinstance(value, (list, tuple, np.ndarray)):

            original = pd.Series(value)

            numeric = pd.to_numeric(
                original,
                errors="coerce"
            ).dropna()

            print("Original length:", len(original))
            print("Numeric values:", len(numeric))
            print("Unique values:", numeric.nunique())

            if len(numeric):

                print("Minimum:", numeric.min())
                print("Maximum:", numeric.max())
                print("Mean:", numeric.mean())

                print(
                    "First 5:",
                    numeric.head(5).tolist()
                )

                print(
                    "Last 5:",
                    numeric.tail(5).tolist()
                )

        else:

            print("Type:", type(value).__name__)
            print("Value:", repr(value)[:300])


# ============================================================
# PART B: INVESTIGATE SESSION FILES AND METADATA
# ============================================================

print("\n" + "=" * 75)
print("PART B: SESSION METADATA INVESTIGATION")
print("=" * 75)

event_dir = find_directory(
    RAW_ROOT / "2026",
    "Australian Grand Prix"
)

session_dir = find_directory(
    event_dir,
    "Race"
)

if session_dir is None:

    print("Australian GP Race directory not found.")

else:

    print("\nSESSION DIRECTORY:")
    print(session_dir)

    print("\nJSON FILES AVAILABLE:")

    for path in sorted(session_dir.glob("*.json")):
        print(path.name)

    # Investigate likely metadata files.
    # Exclude large data files from recursive metadata inspection.

    print("\nPOSSIBLE SESSION START METADATA:")

    for path in sorted(session_dir.glob("*.json")):

        filename = normalize(path.stem)

        if not any(
            word in filename
            for word in [
                "sessioninfo",
                "sessionstatus",
                "sessionmetadata",
                "session",
                "metadata"
            ]
        ):
            continue

        if filename == "sessionlaptimes":
            continue

        try:

            data = read_json(path)

            if isinstance(data, dict):

                print("\nFILE:", path.name)

                print("ROOT KEYS:", list(data.keys()))

                for key, value in data.items():

                    if any(
                        word in key.lower()
                        for word in [
                            "start",
                            "date",
                            "time",
                            "session"
                        ]
                    ):

                        if not isinstance(value, (list, dict)):
                            print(
                                key,
                                "=",
                                repr(value)[:300]
                            )

        except Exception as error:

            print("Could not inspect:", path.name, error)


# ============================================================
# PART C: COMPARE RCM AND LAP TIME FORMATS
# ============================================================

print("\n" + "=" * 75)
print("PART C: TIMESTAMP FORMAT COMPARISON")
print("=" * 75)

if session_dir is not None:

    rcm_path = session_dir / "rcm.json"

    lap_path = session_dir / "session_laptimes.json"

    weather_path = session_dir / "weather.json"

    # Race Control timestamps

    if rcm_path.exists():

        rcm = read_json(rcm_path)

        print("\nRCM TIME FIELD:")

        rcm_times = rcm.get("time", [])

        for value in rcm_times[:5]:

            print(
                "Value:",
                repr(value),
                "| Python type:",
                type(value).__name__
            )

        if rcm_times:

            parsed = pd.to_datetime(
                pd.Series(rcm_times),
                errors="coerce"
            )

            print("\nParsed RCM timestamps:")
            print(parsed.head())

            print("\nRCM timestamp range:")
            print("Minimum:", parsed.min())
            print("Maximum:", parsed.max())

    # Lap session times

    if lap_path.exists():

        lap_data = read_json(lap_path)

        print("\nLAP TIME FIELD:")

        for field in ["time", "sesT", "lap", "drv"]:

            values = lap_data.get(field, [])

            print("\nFIELD:", field)
            print("First 5:", values[:5])
            print("Last 5:", values[-5:])

    # Weather session times

    if weather_path.exists():

        weather = read_json(weather_path)

        print("\nWEATHER TIME FIELD:")

        weather_times = weather.get("wT", [])

        print("First 5:", weather_times[:5])
        print("Last 5:", weather_times[-5:])

        print(
            "Weather time data type:",
            type(weather_times).__name__
        )


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 75)
print("STEP 2.9A.9 COMPLETED")
print("=" * 75)

print("Telemetry structures inspected.")
print("Session metadata inspected.")
print("RCM and lap timestamp formats inspected.")

print("No original JSON files modified.")


PART A: TELEMETRY JSON STRUCTURE

COMPLETE TELEMETRY
File: C:\Users\ayepd\Desktop\For1Sight Project\raw_hf_data\2026\Austrian Grand Prix\Race\SAI\26_tel.json

ROOT JSON KEYS:
['tel']

TELEMETRY OBJECT TYPE:
dict

TELEMETRY FIELD LENGTHS:
time                 | Length: 1155
rpm                  | Length: 1155
speed                | Length: 1155
gear                 | Length: 1155
throttle             | Length: 1155
brake                | Length: 1155
drs                  | Length: 1155
distance             | Length: 1155
rel_distance         | Length: 1155
DriverAhead          | Length: 1155
DistanceToDriverAhead | Length: 1155
acc_x                | Length: 1155
acc_y                | Length: 1155
acc_z                | Length: 1155
x                    | Length: 1155
y                    | Length: 1155
z                    | Length: 1155
dataKey              | Type: str

IMPORTANT NUMERICAL FIELD INVESTIGATION:

FIELD: time
Original length: 1155
Numeric values: 1155
Unique values: 115

In [23]:

# ============================================================
# FOR1SIGHT STEP 2.9A.10
# TELEMETRY CHANNEL INTEGRITY AUDIT
# ============================================================

import json
import re
import pandas as pd
import numpy as np
from pathlib import Path

RAW_ROOT = Path(
    r"C:\Users\ayepd\Desktop\For1Sight Project\raw_hf_data"
)

REPORT_DIR = RAW_ROOT.parent / "audit_reports"

QUALITY_FILE = (
    REPORT_DIR / "step2_9A7_telemetry_validation.csv"
)

OUTPUT_FILE = (
    REPORT_DIR / "step2_9A10_channel_integrity.csv"
)

quality_df = pd.read_csv(QUALITY_FILE)


def normalize(text):
    return re.sub(
        r"[^a-z0-9]",
        "",
        str(text).lower()
    )


def find_directory(parent, name):

    if parent is None or not parent.exists():
        return None

    for folder in parent.iterdir():

        if (
            folder.is_dir()
            and normalize(folder.name) == normalize(name)
        ):
            return folder

    return None


def get_numeric(values):

    return pd.to_numeric(
        pd.Series(values),
        errors="coerce"
    ).dropna()


results = []

print("=" * 75)
print("FOR1SIGHT STEP 2.9A.10")
print("TELEMETRY CHANNEL INTEGRITY AUDIT")
print("=" * 75)


for _, row in quality_df.iterrows():

    season = int(row["Season"]) if "Season" in row.index else 2026

    event_dir = find_directory(
        RAW_ROOT / str(season),
        row["Event"]
    )

    session_dir = find_directory(
        event_dir,
        row["Session"]
    )

    driver_dir = find_directory(
        session_dir,
        row["Driver"]
    )

    if driver_dir is None:
        continue

    lap = int(row["Lap"])

    path = driver_dir / f"{lap}_tel.json"

    if not path.exists():
        continue

    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    tel = raw.get("tel", {})

    if not isinstance(tel, dict):
        continue

    fields = [
        "time",
        "speed",
        "rpm",
        "gear",
        "throttle",
        "brake",
        "drs",
        "distance",
        "rel_distance"
    ]

    stats = {}

    for field in fields:

        values = tel.get(field, [])

        numeric = get_numeric(values)

        stats[field] = {
            "length": len(values) if isinstance(values, list) else 0,
            "numeric": len(numeric),
            "unique": numeric.nunique(),
            "minimum": numeric.min() if len(numeric) else np.nan,
            "maximum": numeric.max() if len(numeric) else np.nan,
            "std": numeric.std(ddof=0) if len(numeric) else np.nan
        }

    speed = stats["speed"]
    rpm = stats["rpm"]
    throttle = stats["throttle"]
    gear = stats["gear"]
    brake = stats["brake"]
    distance = get_numeric(tel.get("distance", []))
    relative = get_numeric(tel.get("rel_distance", []))
    time = get_numeric(tel.get("time", []))

    # Check distance progression

    distance_monotonic = (
        distance.diff().dropna() >= -0.000001
    ).all() if len(distance) > 1 else False

    # Expected channel ranges.
    # These are screening rules, not cleaning transformations.

    speed_valid = (
        speed["minimum"] >= 0
        and speed["maximum"] <= 400
    ) if speed["numeric"] else False

    rpm_valid = (
        rpm["minimum"] >= 0
        and rpm["maximum"] <= 20000
    ) if rpm["numeric"] else False

    throttle_valid = (
        throttle["minimum"] >= 0
        and throttle["maximum"] <= 100
    ) if throttle["numeric"] else False

    gear_valid = (
        gear["minimum"] >= 0
        and gear["maximum"] <= 8
    ) if gear["numeric"] else False

    
    # Validate relative distance

    relative_valid = (
        len(relative) > 0
        and relative.min() >= -0.001
        and relative.max() <= 1.01
    )


    # Detect suspicious constant performance channels.

    constant_channels = []

    for field in ["speed", "rpm", "throttle", "gear"]:

        if (
            stats[field]["numeric"] >= 100
            and stats[field]["unique"] <= 1
        ):
            constant_channels.append(field)

    # Create overall screening status.

    if len(time) < 100:
        quality_status = "INSUFFICIENT_SAMPLES"

    elif (
        constant_channels
        or not speed_valid
        or not rpm_valid
        or not throttle_valid
        or not gear_valid
        or not distance_monotonic
        or not relative_valid
    ):
        quality_status = "REQUIRES_INVESTIGATION"

    else:
        quality_status = "PASSES_INITIAL_SCREENING"

    result = {
        "Season": season,
        "Event": row["Event"],
        "Session": row["Session"],
        "Driver": row["Driver"],
        "Lap": lap,

        "Samples": len(time),

        "DataKey": tel.get("dataKey"),

        "Speed_Unique": speed["unique"],
        "Speed_Min": speed["minimum"],
        "Speed_Max": speed["maximum"],
        "Speed_Std": speed["std"],

        "RPM_Unique": rpm["unique"],
        "RPM_Min": rpm["minimum"],
        "RPM_Max": rpm["maximum"],
        "RPM_Std": rpm["std"],

        "Throttle_Unique": throttle["unique"],
        "Throttle_Min": throttle["minimum"],
        "Throttle_Max": throttle["maximum"],

        "Gear_Unique": gear["unique"],
        "Gear_Min": gear["minimum"],
        "Gear_Max": gear["maximum"],

        "Brake_Unique": brake["unique"],

        "Distance_Monotonic": distance_monotonic,
        "Relative_Distance_Valid": relative_valid,

        "Speed_Valid": speed_valid,
        "RPM_Valid": rpm_valid,
        "Throttle_Valid": throttle_valid,
        "Gear_Valid": gear_valid,

        "Constant_Channels": ", ".join(constant_channels),

        "Quality_Status": quality_status
    }

    results.append(result)


# ============================================================
# DISPLAY RESULTS
# ============================================================

audit_df = pd.DataFrame(results)

print("\nTOTAL TELEMETRY FILES:", len(audit_df))

print("\nQUALITY STATUS DISTRIBUTION:")

print(
    audit_df["Quality_Status"]
    .value_counts(dropna=False)
)

print("\nCHANNEL INTEGRITY RESULTS:")

display(
    audit_df[
        [
            "Event",
            "Session",
            "Driver",
            "Lap",
            "Samples",
            "Speed_Unique",
            "RPM_Unique",
            "Throttle_Unique",
            "Gear_Unique",
            "Constant_Channels",
            "Quality_Status"
        ]
    ]
)

print("\nDATAKEY INVESTIGATION:")

display(
    audit_df[
        [
            "Event",
            "Driver",
            "Lap",
            "DataKey"
        ]
    ]
)

# Save report

audit_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\n" + "=" * 75)
print("STEP 2.9A.10 COMPLETED")
print("=" * 75)

print("Report saved:", OUTPUT_FILE)

print("Original JSON files were not modified.")


FOR1SIGHT STEP 2.9A.10
TELEMETRY CHANNEL INTEGRITY AUDIT

TOTAL TELEMETRY FILES: 26

QUALITY STATUS DISTRIBUTION:
Quality_Status
INSUFFICIENT_SAMPLES      19
REQUIRES_INVESTIGATION     7
Name: count, dtype: int64

CHANNEL INTEGRITY RESULTS:


,Event,Session,Driver,Lap,Samples,Speed_Unique,RPM_Unique,Throttle_Unique,Gear_Unique,Constant_Channels,Quality_Status
0,Australian Grand Prix,Race,BOT,18,3,1,1,1,1,,INSUFFICIENT_SAMPLES
1,Austrian Grand Prix,Race,SAI,26,1155,1,1,1,1,"speed, rpm, throttle, gear",REQUIRES_INVESTIGATION
2,Azerbaijan Grand Prix,Practice 1,HUL,24,1132,1,1,1,1,"speed, rpm, throttle, gear",REQUIRES_INVESTIGATION
3,Azerbaijan Grand Prix,Race,GAS,38,4,1,1,1,1,,INSUFFICIENT_SAMPLES
4,Azerbaijan Grand Prix,Race,NOR,38,3,1,1,1,1,,INSUFFICIENT_SAMPLES
5,Bahrain Grand Prix,Practice 2,BOR,27,4,1,1,1,1,,INSUFFICIENT_SAMPLES
6,Bahrain Grand Prix,Race,ALB,44,3,1,1,1,1,,INSUFFICIENT_SAMPLES
7,Bahrain Grand Prix,Race,BOT,10,3,1,1,1,1,,INSUFFICIENT_SAMPLES
8,Bahrain Grand Prix,Race,RUS,52,3,1,1,1,1,,INSUFFICIENT_SAMPLES
9,Barcelona Grand Prix,Race,ALO,40,3,1,1,1,1,,INSUFFICIENT_SAMPLES



DATAKEY INVESTIGATION:


,Event,Driver,Lap,DataKey
0,Australian Grand Prix,BOT,18,2026-Australian Grand Prix-Race-BOT-18
1,Austrian Grand Prix,SAI,26,2026-Austrian Grand Prix-Race-SAI-26
2,Azerbaijan Grand Prix,HUL,24,2026-Azerbaijan Grand Prix-Practice 1-HUL-24
3,Azerbaijan Grand Prix,GAS,38,2026-Azerbaijan Grand Prix-Race-GAS-38
4,Azerbaijan Grand Prix,NOR,38,2026-Azerbaijan Grand Prix-Race-NOR-38
5,Bahrain Grand Prix,BOR,27,2026-Bahrain Grand Prix-Practice 2-BOR-27
6,Bahrain Grand Prix,ALB,44,2026-Bahrain Grand Prix-Race-ALB-44
7,Bahrain Grand Prix,BOT,10,2026-Bahrain Grand Prix-Race-BOT-10
8,Bahrain Grand Prix,RUS,52,2026-Bahrain Grand Prix-Race-RUS-52
9,Barcelona Grand Prix,ALO,40,2026-Barcelona Grand Prix-Race-ALO-40



STEP 2.9A.10 COMPLETED
Report saved: C:\Users\ayepd\Desktop\For1Sight Project\audit_reports\step2_9A10_channel_integrity.csv
Original JSON files were not modified.
